# Copilot adoption journey and ways-of-working analysis

This end-to-end companion to `copilot-analytics-examples.ipynb` uses a Viva Insights Person Query
to move from basic Copilot metrics to an adoption journey and ways-of-working assessment.
It answers four practical questions:

1. **Reach:** how the licensed Copilot population (where available), metric coverage,
   and weekly activation expanded over time.
2. **Habit:** how many people show sustained use, emerging use, or no use, using Power
   and Habitual User status as the notebook's proxy for *stickiness* - durable, repeated
   return usage rather than a one-off trial.
3. **Opportunity:** where adoption differs across managers, individual contributors, and functions.
4. **Ways of working:** how sustained use is associated with collaboration load, meetings,
   focus, multitasking, and after-hours work.

The analysis is observational. It describes associations and adoption patterns; it does not
claim that Copilot caused the working-pattern differences. If you need to estimate a causal
effect, see the [Causal Inference in Copilot Analytics](https://microsoft.github.io/viva-insights-sample-code/causal-inference/)
page and the [Copilot Causal Toolkit](https://microsoft.github.io/viva-insights-sample-code/copilot-causal-toolkit/).

## Which notebook should I use?

| Notebook | Use it when |
| --- | --- |
| [`copilot-analytics-examples.ipynb`](https://github.com/microsoft/viva-insights-sample-code/blob/main/examples/utility-python/copilot-analytics-examples.ipynb) | You want a guided tour of the core Copilot metrics and the standard **vivainsights** visuals, and you are getting oriented in the data. |
| **This notebook** | You have at least 12 weeks of Copilot data and need an end-to-end adoption assessment: cohorts, habit formation, conversion targeting, and associated ways of working. |

## Before you run it

The notebook is organization-agnostic: it makes no assumptions about a specific
organization's structure, headcount, or function names. Set `INPUT_FILE` in the
configuration cell to your own Person Query export, in CSV or Parquet format.

The export should contain:

- at least 12 weeks of Copilot activity, so that the rolling habit window can be evaluated;
- the collaboration and work-pattern metrics used by the diagnostic sections. Any metric
  that is absent is skipped automatically rather than causing an error; and
- ideally one or more organizational attributes, such as `FunctionType` or `Organization`,
  `IsManager` or `SupervisorIndicator`, and `LevelDesignation` or `Level`.

Segment definitions follow the standard Copilot Usage Segments. For the background on why
the segments are defined the way they are, including the 9-of-12-weeks rationale and what
to do with fewer than three months of data, see the
[Copilot Usage Segments](https://microsoft.github.io/viva-insights-sample-code/copilot-usage-segments/) page.


In [ ]:
from pathlib import Path

# ---------------------------------------------------------------------------
# Input. Replace this with your own Viva Insights Person Query export.
# Both .csv and .parquet are supported. The export needs at least 12 weeks of
# Copilot activity for the rolling habit window to be meaningful.
# ---------------------------------------------------------------------------
INPUT_FILE = Path("person_query.csv")
OUTPUT_DIR = Path("outputs/copilot-adoption-journey")

# ---------------------------------------------------------------------------
# Privacy and group-size floors.
# MIN_PRIVACY_N is the hard floor: no group smaller than this is ever reported.
# MIN_DISPLAY_N is the higher bar used for group comparisons, so that rankings
# are not driven by very small teams.
# ---------------------------------------------------------------------------
MIN_PRIVACY_N = 5
MIN_DISPLAY_N = 30

# ---------------------------------------------------------------------------
# Usage-segment parameters. These are the standard 12-week Copilot Usage Segment
# settings, passed explicitly rather than through the "12w" preset so that
# changing POWER_THRESHOLD actually changes the segmentation. See
# https://microsoft.github.io/viva-insights-sample-code/copilot-usage-segments/
# ---------------------------------------------------------------------------
SEGMENT_WINDOW_WEEKS = 12     # max_window: length of the rolling window
SEGMENT_HABIT_WEEKS = 9       # width: active weeks required within the window
SEGMENT_ACTION_THRESHOLD = 1  # threshold: actions that make a week "active"
POWER_THRESHOLD = 15          # power_thres: mean weekly actions for Power User
RECENT_WEEKS = 4              # trailing window for the process-ratio diagnostics

# Smallest standardised effect worth calling material. With tens of thousands of
# person-weeks, a coefficient can clear the 95% significance bar while being far too
# small to act on, so the diagnostics below judge results on size as well as
# significance. 0.1 standard deviations is the conventional floor for a small effect.
MATERIAL_EFFECT_SD = 0.1

# ---------------------------------------------------------------------------
# Organizational attributes. The notebook looks for each attribute in order and
# uses the first one present in the export, so that it works with the different
# attribute sets that different tenants configure. Add your own column names
# here if they differ.
# ---------------------------------------------------------------------------
FUNCTION_COLUMNS = ["FunctionType", "Organization", "Department"]
MANAGER_COLUMNS = ["IsManager", "SupervisorIndicator", "ManagerIndicator"]
LEVEL_COLUMNS = ["LevelDesignation", "Level"]
LOCATION_COLUMNS = ["Location", "Region"]
LICENSE_COLUMNS = ["Copilot_enabled_days", "Total_Copilot_enabled_days"]

# Functions at or above this Power + Habitual adoption percentage are surfaced as a
# replication playbook. Lower it if no group currently qualifies.
LEADING_FUNCTION_THRESHOLD_PCT = 50.0

# Optional: location values known to be data artifacts, for example a registered or
# mailing address used as a default rather than a genuine work site. Excluded only
# from the Location cut, so people keep their function and manager rows. Leave this
# empty unless such an artifact has been confirmed in your own data.
LOCATION_EXCLUDE = []

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_DIR.resolve()}")


## 1. Method and interpretation guardrails

- One row should represent one person-week.
- Blank rows are removed before analysis.
- When a Copilot enabled-days column is present, it is used for the licensed-population
  view; otherwise, non-null Copilot actions and active-days values are treated as
  **metric coverage**, which is not the same as confirmed licensing.
- Usage segmentation uses the standard **12-week rolling definition**, documented on the
  [Copilot Usage Segments](https://microsoft.github.io/viva-insights-sample-code/copilot-usage-segments/#formal-definitions)
  page and implemented by
  [`identify_usage_segments()`](https://microsoft.github.io/vivainsights/reference/identify_usage_segments.html).
- The first weeks of any panel have incomplete rolling histories. Later coverage cohorts
  may not yet have enough observed weeks to qualify as Habitual or Power Users. If you
  have fewer than three months of data, consider the
  [4-week variation](https://microsoft.github.io/viva-insights-sample-code/copilot-usage-segments/)
  rather than the 12-week definition used here.
- Groups smaller than `MIN_DISPLAY_N` (30 by default) are excluded from group comparisons,
  and nothing below the `MIN_PRIVACY_N` floor (5 by default) is reported at all.
- Adjusted comparisons control for function and manager status when those attributes are
  available, but cannot remove all role or seniority differences.
- Every collaboration metric used below is optional. Whatever is missing from your export
  is reported as unavailable and skipped, rather than causing the notebook to fail.

In [ ]:
import re
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import vivainsights as vi

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 80)

C_NAVY = "#16324F"
C_BLUE = "#2F6B9A"
C_TEAL = "#2A7F83"
C_GOLD = "#B9892D"
C_RED = "#A4473D"
C_GREY = "#7A8288"
C_LIGHT = "#E8EDF1"
C_TEXT = "#202428"

plt.rcParams.update({
    "font.family": ["Segoe UI", "DejaVu Sans", "sans-serif"],
    "font.size": 10.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "text.color": C_TEXT,
    "figure.dpi": 120,
})

TABLES = {}
FIGURES = {}


def save_table(frame, name):
    frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
    TABLES[name] = frame.copy()
    return frame


def save_figure(fig, name):
    fig.savefig(OUTPUT_DIR / f"{name}.png", dpi=180, bbox_inches="tight", facecolor="white")
    FIGURES[name] = fig
    return fig


def add_subtitle(ax, text):
    """Add a small italic subtitle stating the exact metric shown, directly under an
    axis title. Charts without an explicit metric named in the title read ambiguously
    once separated from the surrounding narrative text (e.g. on a slide); this keeps
    the metric definition attached to the chart itself. The offset is in points rather
    than axes fractions so that it stays anchored to the title on tall axes."""
    ax.annotate(text, xy=(0.0, 1.0), xycoords="axes fraction",
                xytext=(0, 20), textcoords="offset points",
                fontsize=8.5, style="italic", color=C_GREY, ha="left", va="bottom")


def pct(num, den):
    return np.nan if not den else 100 * num / den


def safe_ratio(num, den):
    """Percentage that returns NaN instead of inf or a divide-by-zero warning.

    Accepts a scalar or a Series for either argument, so that it can be used both for
    element-wise ratios and for "share of total" calculations.
    """
    num = pd.Series(num) if not np.isscalar(num) else num
    if np.isscalar(den):
        den = np.nan if not den else den
        return 100 * num / den
    den = pd.Series(den)
    return 100 * num / den.where(den > 0)


def lookup(frame, row, column, default=np.nan):
    """Read frame.loc[row, column], returning `default` when either is absent.

    Segments, metrics, and organizational groups are all optional in this notebook:
    a segment can fall below the minimum group size, and a metric can be missing from
    the export entirely. Reading through this helper keeps the narrative sections
    degrading gracefully instead of raising KeyError partway through a run.
    """
    if frame is None or column not in getattr(frame, "columns", []):
        return default
    if row not in frame.index:
        return default
    value = frame.loc[row, column]
    if isinstance(value, pd.Series):
        value = value.iloc[0]
    return default if pd.isna(value) else value


def fmt(value, spec="{:.1f}", missing="not available"):
    """Format a number for narrative text, or say so plainly when it is missing."""
    return missing if value is None or pd.isna(value) else spec.format(value)


def first_available(frame, candidates):
    """Return the first candidate column present in `frame`, otherwise None."""
    for name in candidates:
        if name in frame.columns:
            return name
    return None


def normalise_column(name):
    return re.sub(r"[^0-9A-Za-z]+", "_", str(name)).strip("_")

In [ ]:
if INPUT_FILE.suffix.lower() in {".parquet", ".pq"}:
    raw = pd.read_parquet(INPUT_FILE)
else:
    # import_query() reads a Person Query CSV and cleans the column names.
    raw = vi.import_query(str(INPUT_FILE))

raw_rows = len(raw)
raw.columns = [normalise_column(c) for c in raw.columns]

action_columns = [
    col for col in raw.columns
    if col.startswith("Copilot_actions_taken_in")
]
if "Total_Copilot_actions_taken" not in raw.columns and action_columns:
    raw[action_columns] = raw[action_columns].fillna(0)
    raw["Total_Copilot_actions_taken"] = raw[action_columns].sum(axis=1)

required = {"PersonId", "MetricDate", "Total_Copilot_actions_taken",
            "Total_Copilot_active_days"}
missing = sorted(required - set(raw.columns))
if missing:
    raise ValueError(
        f"Missing required columns: {missing}. This notebook needs a Person Query "
        "export containing Copilot activity metrics."
    )

df = raw.dropna(subset=["PersonId", "MetricDate"]).copy()
df["MetricDate"] = pd.to_datetime(df["MetricDate"], errors="coerce")
df = df.dropna(subset=["MetricDate"])

# Resolve the organizational attributes from whichever columns this export provides.
attribute_sources = {}
for target, candidates in [
    ("FunctionType", FUNCTION_COLUMNS),
    ("ManagerSource", MANAGER_COLUMNS),
    ("LevelDesignation", LEVEL_COLUMNS),
    ("Location", LOCATION_COLUMNS),
]:
    source = first_available(df, [normalise_column(c) for c in candidates])
    attribute_sources[target] = source
    if source is None:
        df[target] = np.nan
    elif source != target:
        df[target] = df[source]

bad_text = {"", "NA", "NULL", "#N/A", "#n/a", "N/A", "nan", "None"}
for col in ["FunctionType", "LevelDesignation", "Location", "ManagerSource"]:
    df[col] = df[col].mask(df[col].astype(str).str.strip().isin(bad_text))


def normalise_manager_status(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().upper()
    if text in {"MANAGER", "MANAGER+", "PEOPLE MANAGER", "YES", "TRUE", "Y", "1"}:
        return "Manager"
    if text in {"IC", "INDIVIDUAL CONTRIBUTOR", "NO", "FALSE", "N", "0"}:
        return "IC"
    return str(value).strip()


df["ManagerStatus"] = df["ManagerSource"].map(normalise_manager_status)

duplicate_person_weeks = int(df.duplicated(["PersonId", "MetricDate"]).sum())
if duplicate_person_weeks:
    raise ValueError(f"Found {duplicate_person_weeks:,} duplicate person-week rows.")

df = df.sort_values(["PersonId", "MetricDate"]).reset_index(drop=True)
df["copilot_observed"] = (
    df["Total_Copilot_actions_taken"].notna()
    & df["Total_Copilot_active_days"].notna()
)
df["copilot_active"] = df["Total_Copilot_actions_taken"].fillna(0) > 0
# Distinct from `copilot_active`, which is actions-based: this flag drives the
# population/licensing/active-use chart series and follows Total_Copilot_active_days.
df["copilot_active_days_flag"] = df["Total_Copilot_active_days"].fillna(0) > 0

# Licensing detection. An enabled-days column is the standard Viva Insights field for a
# confirmed licence day-count, but it is not present in every Person Query export. Where
# it exists, licensing is measured directly as the number of distinct people with more
# than zero enabled days in the week. Where it does not, the notebook falls back to
# non-null Copilot telemetry as a coverage proxy, per the Section 1 guardrails, and
# labels it as such everywhere it is reported.
LICENSE_COL = first_available(df, [normalise_column(c) for c in LICENSE_COLUMNS])
has_license_col = LICENSE_COL is not None
if has_license_col:
    df[LICENSE_COL] = pd.to_numeric(df[LICENSE_COL], errors="coerce")
    df["copilot_licensed"] = df[LICENSE_COL] > 0
else:
    df["copilot_licensed"] = np.nan

blank_rows_removed = raw_rows - len(df)
weeks = np.sort(df["MetricDate"].unique())
latest_week = pd.Timestamp(weeks[-1])
first_week = pd.Timestamp(weeks[0])

if len(weeks) < SEGMENT_WINDOW_WEEKS:
    warnings.warn(
        f"Only {len(weeks)} weeks are available, fewer than the {SEGMENT_WINDOW_WEEKS}-week "
        "rolling window. Habitual and Power User counts will be understated. See "
        "https://microsoft.github.io/viva-insights-sample-code/copilot-usage-segments/ "
        "for the 4-week variation designed for shorter panels."
    )

# Report-wide metadata used to build informative footnotes: the date range covered, and
# population size broken down by total measured population, Copilot-licensed population,
# and Copilot-active population, all as of the latest observed week.
latest_rows = df["MetricDate"] == latest_week
total_population_n = int(df.loc[latest_rows, "PersonId"].nunique())
licensed_population_n = int(
    df.loc[latest_rows & df["copilot_licensed"].fillna(False), "PersonId"].nunique()
) if has_license_col else total_population_n
active_population_n = int(
    df.loc[latest_rows & df["copilot_active_days_flag"], "PersonId"].nunique()
)
report_metadata = pd.DataFrame([{
    "analysis_start": first_week.date().isoformat(),
    "analysis_end": latest_week.date().isoformat(),
    "weeks_covered": len(weeks),
    "total_population_n": total_population_n,
    "licensed_population_n": licensed_population_n,
    "active_population_n": active_population_n,
    "license_column_used": LICENSE_COL if has_license_col else "",
    "licensed_is_assumed": not has_license_col,
}])
save_table(report_metadata, "00_report_metadata")
FOOTNOTE_BASE = (
    f"{report_metadata.at[0, 'analysis_start']} to {report_metadata.at[0, 'analysis_end']} "
    f"({len(weeks)} weeks) | n={total_population_n:,} total, "
    f"{licensed_population_n:,} licensed"
    + ("*" if not has_license_col else "")
    + f", {active_population_n:,} active in latest week"
)
print(f"Footnote base string: {FOOTNOTE_BASE}\n")

print(f"Rows in file:           {raw_rows:,}")
print(f"Blank rows removed:     {blank_rows_removed:,}")
print(f"Analysis rows:          {len(df):,}")
print(f"People:                 {df['PersonId'].nunique():,}")
print(f"Weeks:                  {len(weeks)} "
      f"({first_week.date()} to {latest_week.date()})")
print(f"Duplicate person-weeks: {duplicate_person_weeks:,}")
print("\nOrganizational attributes resolved from this export:")
for target, source in attribute_sources.items():
    label = target if target != "ManagerSource" else "ManagerStatus"
    if source is None:
        print(f"  {label:<18} not available")
    else:
        coverage = df[target if target != 'ManagerSource' else 'ManagerStatus'].notna().mean()
        print(f"  {label:<18} from '{source}' ({coverage * 100:.1f}% populated)")
print(
    f"\nLicence column:         "
    + (f"'{LICENSE_COL}'" if has_license_col
       else "not available, metric coverage is used as a proxy")
)


## 2. Coverage and activation momentum

The null pattern in the Copilot metrics changes over time. Where the data supports it,
the notebook keeps up to three separate measures, from broadest to narrowest:

- **Licensed population** *(shown only when an enabled-days column is present in the
  export)*: people with at least one enabled day in the week, counted as distinct
  `PersonId` per `MetricDate`. This is the earliest and broadest lens, because it
  reflects provisioning rather than usage.
- **Metric coverage:** people with a non-null Copilot actions and active-days value. When
  no enabled-days column exists, this is the best available proxy for licensing.
- **Weekly activation:** covered people with at least one Copilot action.

This distinction matters because a rise in covered people may reflect licence rollout,
query scope, or telemetry completeness. It should not automatically be labelled adoption.

In [ ]:
weekly = (
    df.groupby("MetricDate")
      .agg(
          population=("PersonId", "nunique"),
          copilot_covered=("copilot_observed", "sum"),
          active=("copilot_active", "sum"),
          active_days_covered=("copilot_active_days_flag", "sum"),
          licensed=("copilot_licensed", "sum"),
          total_actions=("Total_Copilot_actions_taken", "sum"),
      )
      .reset_index()
)
weekly["coverage_pct"] = safe_ratio(weekly["copilot_covered"], weekly["population"])
weekly["active_pct_covered"] = safe_ratio(weekly["active"], weekly["copilot_covered"])
weekly["active_pct_population"] = safe_ratio(weekly["active"], weekly["population"])
weekly["actions_per_active"] = (
    weekly["total_actions"] / weekly["active"].where(weekly["active"] > 0)
)
weekly["active_days_covered_pct"] = safe_ratio(
    weekly["active_days_covered"], weekly["population"]
)
if has_license_col:
    weekly["licensed_pct"] = safe_ratio(weekly["licensed"], weekly["population"])
else:
    # No enabled-days column is available. Rather than dropping the licensed tier from
    # the chart entirely, show it as an assumed upper bound equal to the total measured
    # population, clearly styled and labelled as an assumption so that it cannot be
    # mistaken for confirmed licensing.
    weekly["licensed"] = weekly["population"]
    weekly["licensed_pct"] = 100.0

observed = df[df["copilot_observed"]].copy()
first_observed = observed.groupby("PersonId")["MetricDate"].min().rename("first_observed")
newly_observed = first_observed.value_counts().sort_index().rename("newly_observed")
weekly = weekly.merge(newly_observed, left_on="MetricDate", right_index=True, how="left")
weekly["newly_observed"] = weekly["newly_observed"].fillna(0).astype(int)
save_table(weekly, "02_weekly_coverage_and_activation")

# Three side-by-side single-axis panels rather than a two-panel design with a secondary
# (twin) axis, because a dual-axis chart invites misreading when the two y-scales are
# easy to conflate at a glance.
fig, axes = plt.subplots(1, 3, figsize=(19, 4.8))

ax = axes[0]
ax.plot(weekly["MetricDate"], weekly["population"], marker="o", lw=2.0,
        color=C_GREY, linestyle="--", label="Total population (Viva Insights)")
if has_license_col:
    ax.plot(weekly["MetricDate"], weekly["licensed"], marker="o", lw=2.3,
            color=C_GOLD, label=f"Licensed ({LICENSE_COL} > 0)")
else:
    ax.plot(weekly["MetricDate"], weekly["licensed"], lw=2.3, linestyle=":",
            color=C_GOLD, label="Licensed (assumed = full population*)")
ax.plot(weekly["MetricDate"], weekly["active_days_covered"], marker="o", lw=2.3,
        color=C_TEAL, label="Active (Copilot active days > 0)")
ax.set_ylabel("People")
ax.set_ylim(0, weekly["population"].max() * 1.08)
ax.legend(frameon=False, fontsize=8.5, loc="lower left")
ax.set_title("Population, licensing, and active use over time", loc="left",
             fontweight="bold")
add_subtitle(ax, "Distinct PersonId by MetricDate: total population, licensed, and active")
if not has_license_col:
    ax.annotate(
        "*No enabled-days column is available, so licensed is shown as an assumed\n"
        "upper bound equal to total population, not confirmed licensing.",
        xy=(0.01, -0.32), xycoords="axes fraction", fontsize=7.5, color=C_GREY,
    )

ax = axes[1]
new_after_baseline = weekly["newly_observed"].copy()
new_after_baseline.iloc[0] = 0
ax.bar(weekly["MetricDate"], new_after_baseline, width=5.2, color=C_GOLD,
       label="Newly covered")
ax.set_ylabel("Newly covered people")
ax.set_title("New Copilot metric coverage by week", loc="left", fontweight="bold")
add_subtitle(ax, "Count of people with a first-observed Copilot metric date in that week")
ax.grid(axis="x", visible=False)

ax = axes[2]
ax.plot(weekly["MetricDate"], weekly["actions_per_active"], color=C_BLUE,
        marker="o", lw=2.2, label="Actions per active user")
ax.set_ylabel("Actions per active user")
ax.set_title("Usage depth among active users", loc="left", fontweight="bold")
add_subtitle(ax, "Mean total Copilot actions taken per active person, per week")

for ax in axes:
    ax.tick_params(axis="x", rotation=30)

fig.suptitle("Copilot population, licensing, and activation", x=0.01,
             ha="left", fontsize=15, fontweight="bold")
fig.tight_layout()
save_figure(fig, "02_coverage_and_activation")
plt.show()

first_row, last_row = weekly.iloc[0], weekly.iloc[-1]
largest_additions = weekly.iloc[1:].nlargest(3, "newly_observed")

coverage_change = last_row["coverage_pct"] - first_row["coverage_pct"]
depth_change = last_row["actions_per_active"] - first_row["actions_per_active"]
depth_change_pct = (
    100 * depth_change / first_row["actions_per_active"]
    if first_row["actions_per_active"] else np.nan
)

print(
    f"Metric coverage moved from {fmt(first_row['coverage_pct'])}% to "
    f"{fmt(last_row['coverage_pct'])}% ({coverage_change:+.1f} points); active among "
    f"covered moved from {fmt(first_row['active_pct_covered'])}% to "
    f"{fmt(last_row['active_pct_covered'])}%."
)
if has_license_col:
    print(
        f"Licensed population moved from {fmt(first_row['licensed_pct'])}% to "
        f"{fmt(last_row['licensed_pct'])}% of the measured population."
    )
else:
    print(
        "No enabled-days column is present in this export. The licensed series is "
        "shown as an assumed upper bound equal to the total population (dotted line), "
        "not confirmed licensing."
    )
print(
    f"Actions per active user were {fmt(first_row['actions_per_active'])} initially and "
    f"{fmt(last_row['actions_per_active'])} in the latest week "
    f"({fmt(depth_change_pct, '{:+.0f}')}%)."
)
if len(largest_additions):
    print("\nLargest additions to Copilot metric coverage after the baseline week:")
    print(largest_additions[["MetricDate", "newly_observed", "coverage_pct",
                             "active_pct_covered"]].to_string(index=False))
    print(
        "\nCheck what drove these weeks before reading them as adoption: a licence "
        "rollout, a change in query scope, or a change in telemetry completeness will "
        "all look the same here."
    )


## 3. Adoption journey and habit formation

**Why this section matters:** a single week of use does not tell you whether Copilot has
become part of someone's routine. Power and Habitual User status is this notebook's proxy
for **stickiness** - evidence of durable, repeated return usage rather than a one-off
trial. Both segments require at least one action in **9 of the trailing 12 weeks**, so
membership cannot be earned by a single busy week; it requires activity spread across
roughly a quarter. Power Users add a volume threshold on top of that consistency, which
separates heavy, embedded use from lighter-but-still-durable habitual use. Together they
answer a question a simple "used it this week" metric cannot: has the behaviour persisted
long enough to call it a habit, and is it a return habit rather than a trial?

The notebook applies the standard **12-week rolling usage-segment definition** described
on the [Copilot Usage Segments](https://microsoft.github.io/viva-insights-sample-code/copilot-usage-segments/#formal-definitions)
page and implemented by
[`identify_usage_segments()`](https://microsoft.github.io/vivainsights/reference/identify_usage_segments.html).
An "active week" means the target metric records at least one action.

- **Power User:** at least one action in 9 or more of the trailing 12 weeks **and**
  an average of at least 15 weekly actions over the rolling period.
- **Habitual User:** at least one action in 9 or more of the trailing 12 weeks, but the
  rolling weekly average is below the Power User threshold.
- **Novice User:** a rolling average of at least one weekly action, without meeting the
  9-of-12 habit requirement.
- **Low User:** at least one action during the rolling period, but an average below one
  weekly action and without meeting the habit requirement.
- **Non-user:** no actions during the rolling period.

The categories are evaluated in that order, so Power Users are a high-volume subset of
habitual users. Because the package calculates rolling averages from available history,
Novice, Low, and Non-user labels can appear before a person has 12 observed weeks. A person
cannot satisfy the 9-of-12 Habitual or Power requirement without at least nine active weeks.

> **A note on thresholds.** The `version="12w"` preset ignores any `power_thres` you pass
> and always applies 15. To keep the configuration cell honest, this notebook calls
> `identify_usage_segments()` with `version=None` and supplies `threshold`, `width`,
> `max_window`, and `power_thres` explicitly. That reproduces the standard 12-week
> definition exactly at the default settings, and it means that changing
> `POWER_THRESHOLD` genuinely changes the segmentation rather than only relabelling it.

For executive interpretation, Power and Habitual are combined as **Power + Habitual
Users**, Novice and Low as **Emerging**, with Non-user retained separately. A plain-English glossary
of every segment, saved as `00_usage_segment_definitions`, is exported below as a
standalone asset so the definitions can be dropped directly into decks and reports.

In [ ]:
habit_rule = (
    f"Active in at least {SEGMENT_HABIT_WEEKS} of the trailing "
    f"{SEGMENT_WINDOW_WEEKS} weeks"
)
active_week_rule = (
    f"a week counts as active at {SEGMENT_ACTION_THRESHOLD}+ Copilot action(s)"
)

segment_definitions = pd.DataFrame([
    {
        "segment": "Power User",
        "definition": (
            f"{habit_rule} AND an average of at least {POWER_THRESHOLD} weekly actions "
            f"over that window ({active_week_rule})."
        ),
        "what_it_indicates": (
            "The highest-stickiness usage: frequent return visits at high volume. The "
            "clearest evidence that Copilot has become embedded in someone's workflow."
        ),
    },
    {
        "segment": "Habitual User",
        "definition": (
            f"{habit_rule}, with a rolling weekly average below the Power User "
            f"threshold of {POWER_THRESHOLD}."
        ),
        "what_it_indicates": (
            "A durable weekly habit has formed even though usage volume is modest, so "
            "consistency rather than intensity is the signal."
        ),
    },
    {
        "segment": "Novice User",
        "definition": (
            f"A rolling average of at least one weekly action, without meeting the "
            f"{SEGMENT_HABIT_WEEKS}-of-{SEGMENT_WINDOW_WEEKS} week requirement."
        ),
        "what_it_indicates": (
            "Has tried Copilot repeatedly but has not yet formed a consistent weekly "
            "habit, so this is the main pool for conversion into Power + Habitual use."
        ),
    },
    {
        "segment": "Low User",
        "definition": (
            "At least one action in the rolling period, but a rolling average below one "
            "weekly action and not meeting the habit requirement."
        ),
        "what_it_indicates": "Sporadic, occasional use only.",
    },
    {
        "segment": "Non-user",
        "definition": (
            f"No Copilot actions recorded during the rolling {SEGMENT_WINDOW_WEEKS}-week "
            "period."
        ),
        "what_it_indicates": "No observed usage.",
    },
])
save_table(segment_definitions, "00_usage_segment_definitions")
print(segment_definitions.to_string(index=False))


In [ ]:
# Preserve calendar weeks after first observed Copilot telemetry. Dropping
# null rows would compress the rolling window into the last N observed rows.
seg_input = df.merge(first_observed, on="PersonId", how="inner")
seg_input = seg_input[seg_input["MetricDate"] >= seg_input["first_observed"]].copy()
seg_input["Total_Copilot_actions_taken"] = (
    seg_input["Total_Copilot_actions_taken"].fillna(0).astype(float)
)

# version=None with explicit parameters, so that the configured POWER_THRESHOLD is
# actually applied. At the default settings this reproduces the standard "12w"
# definition exactly. See the note in the section above.
segment_kwargs = dict(
    metric="Total_Copilot_actions_taken",
    version=None,
    threshold=SEGMENT_ACTION_THRESHOLD,
    width=SEGMENT_HABIT_WEEKS,
    max_window=SEGMENT_WINDOW_WEEKS,
    power_thres=POWER_THRESHOLD,
)

seg = vi.identify_usage_segments(seg_input.copy(), return_type="data", **segment_kwargs)
segment_col = next(
    col for col in ("UsageSegments", f"UsageSegments_{SEGMENT_WINDOW_WEEKS}w")
    if col in seg.columns
)
seg = seg.rename(columns={segment_col: "UsageSegment_12w"})

latest_segments = (
    seg[seg["MetricDate"] == latest_week][["PersonId", "UsageSegment_12w"]]
    .drop_duplicates("PersonId")
)
journey_map = {
    "Power User": "Power + Habitual",
    "Habitual User": "Power + Habitual",
    "Novice User": "Emerging",
    "Low User": "Emerging",
    "Non-user": "Non-user",
}
latest_segments["JourneyStage"] = latest_segments["UsageSegment_12w"].map(journey_map)

segment_order = ["Power User", "Habitual User", "Novice User", "Low User", "Non-user"]
# Power, Habitual, and Novice match vivainsights.identify_usage_segments()'s own default
# plot colours, so the charts here stay visually consistent with the package's native
# chart. The package renders Low User ("#808080") and Non-user ("grey") as effectively
# the same grey, which is indistinguishable on a slide, so Low User is given a distinct
# gold tone and all five segments remain separable.
segment_color_map = {
    "Power User": "#0c336e",
    "Habitual User": "#1c66b0",
    "Novice User": "#80baea",
    "Low User": "#B9892D",
    "Non-user": "#808080",
}
segment_colors = [segment_color_map[segment] for segment in segment_order]

# A single, shared colour scheme for the 3-stage journey grouping (Power + Habitual /
# Emerging / Non-user), reused consistently across every chart that shows it.
journey_stage_order = ["Power + Habitual", "Emerging", "Non-user"]
journey_stage_color_map = {
    "Power + Habitual": segment_color_map["Power User"],
    "Emerging": segment_color_map["Novice User"],
    "Non-user": segment_color_map["Non-user"],
}
segment_summary = (
    latest_segments["UsageSegment_12w"].value_counts()
    .reindex(segment_order, fill_value=0)
    .rename_axis("segment").reset_index(name="people")
)
segment_summary["pct"] = safe_ratio(
    segment_summary["people"], segment_summary["people"].sum()
)
save_table(segment_summary, "03_latest_usage_segments")

person_journey = (
    observed.groupby("PersonId")
    .agg(
        first_observed=("MetricDate", "min"),
        observed_weeks=("MetricDate", "nunique"),
        active_weeks=("copilot_active", "sum"),
        total_actions=("Total_Copilot_actions_taken", "sum"),
    )
    .reset_index()
    .merge(latest_segments, on="PersonId", how="left")
)
person_journey["active_share"] = (
    person_journey["active_weeks"] / person_journey["observed_weeks"]
)

cohort_latest = (
    person_journey.groupby("first_observed")
    .agg(
        people=("PersonId", "size"),
        active_latest=("PersonId", lambda ids: int(
            df[(df["MetricDate"] == latest_week)
               & (df["PersonId"].isin(ids))
               & df["copilot_active"]]["PersonId"].nunique()
        )),
        power_habitual_latest=("JourneyStage", lambda s: int((s == "Power + Habitual").sum())),
    )
    .reset_index()
)
cohort_latest["active_latest_pct"] = safe_ratio(
    cohort_latest["active_latest"], cohort_latest["people"]
)
cohort_latest["power_habitual_latest_pct"] = safe_ratio(
    cohort_latest["power_habitual_latest"], cohort_latest["people"]
)
cohort_latest["weeks_available"] = (
    (latest_week - cohort_latest["first_observed"]).dt.days // 7 + 1
)
cohort_latest["power_habitual_latest_pct_eligible"] = cohort_latest[
    "power_habitual_latest_pct"
].where(cohort_latest["weeks_available"] >= SEGMENT_WINDOW_WEEKS)
cohort_latest = cohort_latest.sort_values("first_observed").reset_index(drop=True)
save_table(cohort_latest, "03_entry_cohort_journey")


# The package-native time-series view and table, as the source-of-truth presentation
# of the segment calculation.
native_segment_table = vi.identify_usage_segments(
    seg_input.copy(), return_type="table", **segment_kwargs
).reset_index()
save_table(native_segment_table, "03_usage_segments_over_time")

native_segment_fig = vi.identify_usage_segments(
    seg_input.copy(), return_type="plot", **segment_kwargs
)
native_segment_ax = native_segment_fig.axes[0]
native_segment_ax.set_title("", loc="center")
native_segment_ax.set_title(
    f"{SEGMENT_WINDOW_WEEKS}-week Copilot usage segments over time",
    loc="left", fontweight="bold",
)
for annotation in native_segment_ax.texts:
    if annotation.get_text().startswith("Usage Segments"):
        annotation.set_visible(False)
for container in native_segment_ax.containers:
    segment = container.get_label()
    if segment in segment_color_map:
        for patch in container.patches:
            patch.set_facecolor(segment_color_map[segment])
            patch.set_edgecolor("white")
add_subtitle(native_segment_ax,
             "vi.identify_usage_segments(): rolling share of PersonId by usage segment")
native_segment_ax.legend(title="Usage Segment", frameon=True)
native_segment_fig.text(
    0.01, -0.01,
    f"The first {SEGMENT_WINDOW_WEEKS - 1} dates have incomplete "
    f"{SEGMENT_WINDOW_WEEKS}-week histories. Habitual and Power status still require at "
    f"least {SEGMENT_HABIT_WEEKS} active weeks; Novice, Low and Non-user can be assigned "
    "from available history.",
    fontsize=8.5, color=C_GREY,
)
save_figure(native_segment_fig, "03_usage_segments_over_time")
plt.show()

# Entry cohorts answer a different question from the native segment trend:
# whether newly covered people have had time to activate and form a habit.
plot_cohorts = cohort_latest[cohort_latest["people"] >= MIN_DISPLAY_N].copy()
if len(plot_cohorts):
    fig, ax = plt.subplots(figsize=(10.5, 4.8))
    ax.bar(plot_cohorts["first_observed"], plot_cohorts["active_latest_pct"],
           width=5.0, color=C_TEAL, label="Active in latest week")
    ax.plot(plot_cohorts["first_observed"],
            plot_cohorts["power_habitual_latest_pct_eligible"],
            marker="o", lw=2.2, color=journey_stage_color_map["Power + Habitual"],
            label="Power + Habitual at latest week")
    ax.set_ylim(0, 105)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.set_title("Latest-week activation and habit status by entry cohort",
                 loc="left", fontweight="bold")
    add_subtitle(ax, "% of each first-observed-week cohort active or Power + Habitual in the latest week")
    ax.set_xlabel("First week with Copilot metric coverage")
    ax.set_ylabel("% of cohort")
    ax.legend(frameon=False)
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    save_figure(fig, "03_entry_cohort_journey")
    plt.show()
else:
    print(
        f"No entry cohort reaches the {MIN_DISPLAY_N}-person display floor, so the "
        "cohort chart is skipped. This is normal when coverage arrived gradually."
    )

print(segment_summary.to_string(index=False, formatters={"pct": "{:.1f}%".format}))

if len(plot_cohorts):
    print(f"\nEntry cohorts with at least {MIN_DISPLAY_N} people:")
    print(plot_cohorts[[
        "first_observed", "people", "weeks_available", "active_latest_pct",
        "power_habitual_latest_pct_eligible",
    ]].to_string(index=False, formatters={
        "active_latest_pct": "{:.1f}%".format,
        "power_habitual_latest_pct_eligible": lambda v: "not yet eligible" if pd.isna(v)
        else f"{v:.1f}%",
    }))

    eligible_cohorts = plot_cohorts.dropna(subset=["power_habitual_latest_pct_eligible"])
    if len(eligible_cohorts) >= 2:
        oldest, newest = eligible_cohorts.iloc[0], eligible_cohorts.iloc[-1]
        gap = oldest["active_latest_pct"] - newest["active_latest_pct"]
        direction = ("lower" if gap > 0 else "higher") if abs(gap) >= 1 else "similar"
        print(
            f"\nActivation in the newest fully eligible cohort is {direction} than in "
            f"the earliest ({fmt(newest['active_latest_pct'])}% vs "
            f"{fmt(oldest['active_latest_pct'])}%)."
        )


## 4. Where adoption is leading or lagging

`vi.create_rank()` provides the package-native organizational comparison. Because usage
segments are categorical, the ranking uses three numeric adoption measures:

- **Power and Habitual Users (%):** the mean of a 0/100 Habitual-or-Power indicator.
- **Active-week share:** the percentage of observed weeks with at least one Copilot action.
- **Average weekly Copilot actions:** usage depth across observed weeks.

The native dumbbell plot shows the highest and lowest qualifying group for each attribute;
the exported tables retain every qualifying group. Whichever of function, manager status,
and location your export provides is included, at a minimum group size of
`MIN_DISPLAY_N`. Attributes that are missing, or that resolve to a single group, are
skipped automatically.

In [ ]:
latest_attributes = (
    df[df["MetricDate"] == latest_week][
        ["PersonId", "FunctionType", "ManagerStatus", "Location", "LevelDesignation"]
    ]
    .drop_duplicates("PersonId")
)
adoption_snapshot = latest_segments.merge(latest_attributes, on="PersonId", how="left")
adoption_snapshot = adoption_snapshot.merge(
    df[df["MetricDate"] == latest_week][
        ["PersonId", "copilot_active", "Total_Copilot_actions_taken"]
    ].drop_duplicates("PersonId"),
    on="PersonId", how="left"
)


def adoption_by(frame, attribute):
    """Adoption summary for one organizational attribute, or None when unusable."""
    if attribute not in frame.columns or frame[attribute].notna().sum() == 0:
        return None
    summary = (
        frame.dropna(subset=[attribute])
        .groupby(attribute)
        .agg(
            people=("PersonId", "nunique"),
            active=("copilot_active", "sum"),
            power_habitual=("JourneyStage", lambda s: int((s == "Power + Habitual").sum())),
            median_actions_active=("Total_Copilot_actions_taken",
                                   lambda s: s[s > 0].median()),
        )
        .reset_index()
    )
    # Never report a group below the hard privacy floor.
    summary = summary[summary["people"] >= MIN_PRIVACY_N]
    if summary.empty:
        return None
    summary["active_pct"] = safe_ratio(summary["active"], summary["people"])
    summary["power_habitual_pct"] = safe_ratio(
        summary["power_habitual"], summary["people"]
    )
    return summary


manager_summary = adoption_by(adoption_snapshot, "ManagerStatus")
has_manager_split = (
    manager_summary is not None and manager_summary["ManagerStatus"].nunique() > 1
)
if manager_summary is not None:
    save_table(manager_summary, "04_manager_adoption")

function_summary = adoption_by(adoption_snapshot, "FunctionType")
if function_summary is not None:
    function_summary = (
        function_summary[function_summary["people"] >= MIN_DISPLAY_N]
        .sort_values("active_pct", ascending=False)
        .reset_index(drop=True)
    )
    if function_summary.empty:
        function_summary = None
if function_summary is not None:
    save_table(function_summary, "04_function_adoption")

# Where the Novice Users are concentrated. This identifies practical targets for
# conversion outreach, rather than an abstract conversion rate with no action attached.
novice_by_function = None
if function_summary is not None:
    novice_by_function = (
        adoption_snapshot[adoption_snapshot["UsageSegment_12w"] == "Novice User"]
        .dropna(subset=["FunctionType"])
        .groupby("FunctionType")["PersonId"].nunique()
        .rename("novice_people").reset_index()
        .merge(function_summary[["FunctionType", "people"]], on="FunctionType", how="right")
    )
    novice_by_function["novice_people"] = (
        novice_by_function["novice_people"].fillna(0).astype(int)
    )
    novice_by_function["novice_pct_of_function"] = safe_ratio(
        novice_by_function["novice_people"], novice_by_function["people"]
    )
    novice_by_function["share_of_all_novices"] = safe_ratio(
        novice_by_function["novice_people"], novice_by_function["novice_people"].sum()
    )
    novice_by_function = novice_by_function.sort_values(
        "novice_people", ascending=False
    ).reset_index(drop=True)
    save_table(novice_by_function, "04_novice_users_by_function")

rank_data = (
    person_journey
    .merge(latest_attributes[["PersonId", "FunctionType", "ManagerStatus", "Location"]],
           on="PersonId", how="left")
)
rank_data["MetricDate"] = latest_week
rank_data["Power and Habitual Users (%)"] = (
    rank_data["JourneyStage"] == "Power + Habitual"
).astype(float) * 100
rank_data["Active_week_share_pct"] = rank_data["active_share"] * 100
rank_data["Average_weekly_Copilot_actions"] = (
    rank_data["total_actions"] / rank_data["observed_weeks"]
)
if LOCATION_EXCLUDE:
    rank_data["Location"] = rank_data["Location"].where(
        ~rank_data["Location"].isin(LOCATION_EXCLUDE)
    )

# create_rank() needs at least one group clearing MIN_DISPLAY_N, so only pass attributes
# that can actually produce one. This keeps the section working on exports that carry
# only some of the organizational attributes.
rank_hrvars = []
for attribute in ["FunctionType", "ManagerStatus", "Location"]:
    values = rank_data[attribute].dropna()
    if values.nunique() < 1:
        continue
    if values.value_counts().max() < MIN_DISPLAY_N:
        continue
    rank_hrvars.append(attribute)

rank_all_measures = None
rank_sustained = None
if not rank_hrvars:
    print(
        "No organizational attribute in this export has a group of at least "
        f"{MIN_DISPLAY_N} people, so the create_rank() comparisons are skipped. "
        "Add FunctionType, IsManager/SupervisorIndicator, or Location to the query "
        "to enable this section."
    )
else:
    rank_ready = rank_data.dropna(subset=rank_hrvars, how="all").copy()
    for attribute in rank_hrvars:
        rank_ready[attribute] = rank_ready[attribute].fillna("Unknown")

    measures = {
        "Power and Habitual Users (%)": "Power and Habitual Users (%)",
        "Active_week_share_pct": "Active-week share",
        "Average_weekly_Copilot_actions": "Average weekly Copilot actions",
    }
    rank_tables = {}
    for metric, label in measures.items():
        rank_tables[metric] = vi.create_rank(
            rank_ready, metric=metric, hrvar=rank_hrvars,
            mingroup=MIN_DISPLAY_N, return_type="table",
        )
    rank_sustained = rank_tables["Power and Habitual Users (%)"]

    # Preserve the complete table-return outputs, not only the extrema shown in
    # create_rank(return_type="plot").
    rank_all_measures = pd.concat(
        [table.assign(measure=measures[metric]) for metric, table in rank_tables.items()],
        ignore_index=True,
    )[["measure", "hrvar", "attributes", "metric", "n"]].rename(columns={
        "hrvar": "organizational_attribute",
        "attributes": "group",
        "metric": "value",
        "n": "people",
    })
    save_table(rank_all_measures, "04_create_rank_all_measures")

    function_rank = rank_sustained[rank_sustained["hrvar"] == "FunctionType"]
    rank_table_view = pd.concat([
        function_rank.head(10),
        function_rank.tail(10),
        rank_sustained[rank_sustained["hrvar"] != "FunctionType"],
    ]).drop_duplicates(["hrvar", "attributes"]).reset_index(drop=True)
    rank_table_view = rank_table_view.rename(columns={
        "hrvar": "Attribute",
        "attributes": "Group",
        "metric": "Power + Habitual Users (%)",
        "n": "People",
    })
    save_table(rank_table_view, "04_create_rank_leadership_table")

    display(
        rank_table_view.style
        .format({"Power + Habitual Users (%)": "{:.1f}%"})
        .background_gradient(
            subset=["Power + Habitual Users (%)"], cmap="Blues", vmin=0, vmax=100
        )
        .hide(axis="index")
        .set_caption(
            "create_rank(return_type='table'): Power + Habitual adoption by organizational group"
        )
    )

    rank_fig = vi.create_rank(
        rank_ready, metric="Power and Habitual Users (%)", hrvar=rank_hrvars,
        mingroup=MIN_DISPLAY_N, return_type="plot", figsize=(9.5, 5.2),
    )
    # The package titles the plot from the metric column name, and that name already
    # reads cleanly ("Power and Habitual Users (%)"), so no override is needed.
    rank_ax = rank_fig.axes[0]
    for y_pos, hrvar in enumerate(rank_hrvars):
        group_rank = rank_sustained[rank_sustained["hrvar"] == hrvar]
        if group_rank.empty:
            continue
        high, low = group_rank.iloc[0], group_rank.iloc[-1]
        rank_ax.annotate(
            f"{low['attributes']} ({low['metric']:.1f}%)",
            (low["metric"], y_pos), xytext=(-6, -18), textcoords="offset points",
            ha="right", fontsize=7.5, color=C_RED,
        )
        rank_ax.annotate(
            f"{high['attributes']} ({high['metric']:.1f}%)",
            (high["metric"], y_pos), xytext=(6, 8), textcoords="offset points",
            ha="left", fontsize=7.5, color=journey_stage_color_map["Power + Habitual"],
        )
    add_subtitle(rank_ax, "vi.create_rank(): % of PersonId classified Power or Habitual User in latest week, by group")
    save_figure(rank_fig, "04_adoption_rank")
    plt.show()

# The manager segment mix shows the full adoption curve rather than only the highest
# and lowest values returned by the native rank plot.
if has_manager_split:
    segment_manager_mix = pd.crosstab(
        adoption_snapshot["ManagerStatus"],
        adoption_snapshot["UsageSegment_12w"],
        normalize="index",
    ).mul(100).reindex(columns=segment_order, fill_value=0)

    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    segment_manager_mix.plot(
        kind="bar", stacked=True, ax=ax, color=segment_colors, width=0.65
    )
    ax.set_ylabel("% within manager group")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.set_xlabel("")
    ax.set_title("Usage segment mix by manager status", loc="left", fontweight="bold")
    add_subtitle(ax, "% of PersonId in each usage segment, by manager status, in the latest week")
    ax.legend(frameon=False, fontsize=8, ncol=3)
    ax.tick_params(axis="x", rotation=0)
    fig.tight_layout()
    save_figure(fig, "04_manager_segment_mix")
    plt.show()
else:
    segment_manager_mix = None
    print(
        "Manager status is unavailable or has only one value in this export, so the "
        "manager comparison is skipped."
    )

if rank_all_measures is not None:
    print("\nFull create_rank table outputs are exported in 04_create_rank_all_measures.csv.")
    print("\nHighest Power + Habitual adoption groups from create_rank():")
    print(rank_sustained.head(12).to_string(index=False))

if manager_summary is not None:
    print("\nManager and individual contributor adoption:")
    print(manager_summary[["ManagerStatus", "people", "active_pct", "power_habitual_pct",
                           "median_actions_active"]].to_string(index=False, formatters={
        "active_pct": "{:.1f}%".format,
        "power_habitual_pct": "{:.1f}%".format,
        "median_actions_active": "{:.1f}".format,
    }))

if function_summary is not None:
    print(f"\nFunctions with at least {MIN_DISPLAY_N} people:")
    print(function_summary[["FunctionType", "people", "active_pct", "power_habitual_pct",
                            "median_actions_active"]].to_string(index=False, formatters={
        "active_pct": "{:.1f}%".format,
        "power_habitual_pct": "{:.1f}%".format,
        "median_actions_active": "{:.1f}".format,
    }))


### Where to replicate, and where to target

Two practical follow-ups to the ranking above.

**Where to replicate.** Functions that have already crossed a high Power + Habitual
adoption threshold are proof, using this organization's own tooling, policies, and
workload, that habitual use is achievable at scale. Treat them as a replication playbook,
not a league table.

**Where to target.** Novice Users have tried Copilot repeatedly without settling into a
weekly habit, so they are the population closest to converting. Knowing which functions
hold most of them turns the gap into a specific, addressable audience.


In [ ]:
leading_functions = None
if function_summary is not None:
    leading_functions = function_summary[
        function_summary["power_habitual_pct"] >= LEADING_FUNCTION_THRESHOLD_PCT
    ].sort_values("power_habitual_pct", ascending=False).reset_index(drop=True)
    save_table(leading_functions, "04_leading_functions_playbook")

if leading_functions is None:
    print(
        "No function attribute is available in this export, so the leading-function "
        "view is skipped."
    )
elif len(leading_functions):
    fig, ax = plt.subplots(figsize=(9.5, max(3.2, 0.5 * len(leading_functions) + 1.2)))
    bars = ax.barh(leading_functions["FunctionType"], leading_functions["power_habitual_pct"],
                    color=journey_stage_color_map["Power + Habitual"], height=0.6)
    ax.invert_yaxis()
    ax.axvline(LEADING_FUNCTION_THRESHOLD_PCT, color=C_GREY, lw=1, linestyle="--")
    ax.set_xlabel("Power + Habitual Users (%)")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    ax.set_xlim(0, 105)
    ax.set_title(
        f"Functions at or above {LEADING_FUNCTION_THRESHOLD_PCT:.0f}% Power + Habitual "
        "adoption", loc="left", fontweight="bold",
    )
    add_subtitle(ax, f"% of PersonId classified Power or Habitual User, by function, n >= {MIN_DISPLAY_N} per group")
    for bar, value in zip(bars, leading_functions["power_habitual_pct"]):
        ax.annotate(f"{value:.1f}%", (bar.get_width(), bar.get_y() + bar.get_height() / 2),
                    xytext=(6, 0), textcoords="offset points", va="center", fontsize=9)
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    save_figure(fig, "04_leading_functions_playbook")
    plt.show()
    print(
        f"{len(leading_functions)} function(s) have already crossed "
        f"{LEADING_FUNCTION_THRESHOLD_PCT:.0f}% Power + Habitual adoption:"
    )
    print(leading_functions[["FunctionType", "people", "power_habitual_pct"]].to_string(
        index=False, formatters={"power_habitual_pct": "{:.1f}%".format}
    ))
else:
    best = function_summary.sort_values("power_habitual_pct", ascending=False).iloc[0]
    print(
        f"No function has yet crossed {LEADING_FUNCTION_THRESHOLD_PCT:.0f}% Power + "
        f"Habitual adoption. The current leader is {best['FunctionType']} at "
        f"{fmt(best['power_habitual_pct'])}%. Lower LEADING_FUNCTION_THRESHOLD_PCT in "
        "the configuration cell to inspect a wider set of groups."
    )


In [ ]:
total_people = int(segment_summary["people"].sum())
novice_n = int(segment_summary.set_index("segment").loc["Novice User", "people"])

# Where the Novice Users actually are: a concrete view for targeted conversion, showing
# both the count of Novice Users per function and what share of the function they are.
top_novice_functions = (
    novice_by_function[novice_by_function["novice_people"] > 0].head(10)
    if novice_by_function is not None else None
)
if top_novice_functions is not None and len(top_novice_functions):
    fig, ax = plt.subplots(figsize=(9.5, max(3.2, 0.5 * len(top_novice_functions) + 1.2)))
    bars = ax.barh(top_novice_functions["FunctionType"], top_novice_functions["novice_people"],
                    color=segment_color_map["Novice User"], height=0.6)
    ax.invert_yaxis()
    ax.set_xlabel("Novice Users (people)")
    ax.set_title("Novice Users by function", loc="left", fontweight="bold")
    add_subtitle(ax, f"Count of PersonId classified Novice User, by function, n >= {MIN_DISPLAY_N} per group")
    ax.set_ylabel("")
    for bar, row in zip(bars, top_novice_functions.itertuples()):
        ax.annotate(f"{row.novice_people:,} ({row.novice_pct_of_function:.0f}% of function)",
                    (bar.get_width(), bar.get_y() + bar.get_height() / 2),
                    xytext=(6, 0), textcoords="offset points", va="center", fontsize=8.5)
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    save_figure(fig, "04_novice_users_by_function")
    plt.show()

    top_share = top_novice_functions["share_of_all_novices"].head(3).sum()
    print(
        f"Novice population: {novice_n:,} people "
        f"({fmt(pct(novice_n, total_people))}% of the measured population).\n"
    )
    print(
        f"The three largest functions by Novice User count hold "
        f"{fmt(top_share, '{:.0f}')}% of all Novice Users, so they are the most "
        "efficient place to start conversion outreach:"
    )
    print(top_novice_functions[["FunctionType", "novice_people", "novice_pct_of_function",
                                "share_of_all_novices"]].to_string(index=False, formatters={
        "novice_pct_of_function": "{:.1f}%".format,
        "share_of_all_novices": "{:.1f}%".format,
    }))
else:
    print(
        f"Novice population: {novice_n:,} people "
        f"({fmt(pct(novice_n, total_people))}% of the measured population)."
    )
    print(
        "No function attribute is available, or no function has Novice Users above the "
        "reporting floor, so the targeting view is skipped."
    )


## 5. Collaboration and working-pattern profile by usage segment

`vi.keymetrics_scan()` is used as the primary descriptive comparison across Power,
Habitual, Novice, Low, and Non-user segments.

To keep the comparison like-for-like:

- only people with a full rolling window of observed Copilot weeks are included;
- each metric is first averaged to one row per person over that trailing window;
- an explicit metric list is supplied for reproducibility, and any metric your export
  does not contain is dropped from the list rather than causing an error; and
- the minimum segment size is `MIN_DISPLAY_N`, so a segment that is too small to report
  is excluded from the heatmap and from the narrative that follows.

The heatmap is normalized **within each metric row**. Colour indicates which segment is
relatively high or low for that metric, not whether the result is inherently favourable.

A note on two metric names that are easy to misread:

- `Collaboration_span` is an hours-based work-session metric, defined by Microsoft as the
  number of hours spent in work sessions before, during, and after working hours. It is
  relabelled below as **Work session span hours** so that it is not mistaken for network
  breadth.
- Collaboration-network metrics such as internal network size, external network size,
  diverse ties, and strong ties are only included if your Person Query happens to contain
  them. The cell below reports exactly which metrics were found.

See the [Microsoft Viva Insights metric reference](https://learn.microsoft.com/en-us/viva/insights/advanced/reference/metrics)
for the full definitions.


In [ ]:
candidate_scan_metrics = [
    "Collaboration_hours",
    "Collaboration_span",
    "Active_connected_hours",
    "Meetings",
    "Meeting_hours",
    "Calls",
    "Call_hours",
    "Chats_sent",
    "Chat_hours",
    "Emails_sent",
    "Email_hours",
    "Multitasking_hours",
    "Available_to_focus_hours",
    "Uninterrupted_hours",
    "Interrupted_hours",
    "After_hours_collaboration_hours",
    "Time_with_leadership",
    "Internal_network_size",
    "External_network_size",
]
scan_metrics = [metric for metric in candidate_scan_metrics if metric in df.columns]
missing_scan_metrics = [m for m in candidate_scan_metrics if m not in df.columns]

mature_ids = set(
    person_journey.loc[
        person_journey["observed_weeks"] >= SEGMENT_WINDOW_WEEKS, "PersonId"
    ]
)
trailing_start = latest_week - pd.Timedelta(weeks=SEGMENT_WINDOW_WEEKS - 1)
scan_data = (
    df[df["PersonId"].isin(mature_ids) & (df["MetricDate"] >= trailing_start)]
    .groupby("PersonId")[scan_metrics]
    .mean()
    .reset_index()
    .merge(latest_segments[["PersonId", "UsageSegment_12w"]],
           on="PersonId", how="inner")
)
scan_data = scan_data.rename(
    columns={"Collaboration_span": "Work_session_span_hours"}
)
scan_metrics = [
    "Work_session_span_hours" if metric == "Collaboration_span" else metric
    for metric in scan_metrics
]
scan_data["MetricDate"] = latest_week
scan_data["UsageSegment_12w"] = pd.Categorical(
    scan_data["UsageSegment_12w"], categories=segment_order, ordered=True
)

segment_scan_table = None
reported_segments = []
if not scan_metrics:
    print(
        "None of the collaboration metrics used by this section are present in the "
        "export, so the key-metrics scan is skipped."
    )
elif scan_data.empty:
    print(
        f"No person has a full {SEGMENT_WINDOW_WEEKS}-week observed history, so the "
        "like-for-like key-metrics scan is skipped. Re-run with a longer export."
    )
else:
    segment_scan_table = vi.keymetrics_scan(
        scan_data,
        hrvar="UsageSegment_12w",
        mingroup=MIN_DISPLAY_N,
        metrics=scan_metrics,
        return_type="table",
    )
    save_table(segment_scan_table, "05_keymetrics_by_segment")
    reported_segments = list(segment_scan_table["UsageSegment_12w"])

    segment_scan_fig = vi.keymetrics_scan(
        scan_data,
        hrvar="UsageSegment_12w",
        mingroup=MIN_DISPLAY_N,
        metrics=scan_metrics,
        return_type="plot",
        low_color="#DCE6EE",
        mid_color="#F4E4BD",
        high_color="#C86B45",
        textsize=8.5,
        plot_row_scaling_factor=0.52,
    )
    for figure_text in segment_scan_fig.texts:
        if figure_text.get_text().startswith("Data from"):
            figure_text.set_text(
                f"Person-level weekly averages from {trailing_start.date()} to "
                f"{latest_week.date()}; people with at least {SEGMENT_WINDOW_WEEKS} "
                "observed Copilot weeks."
            )
    save_figure(segment_scan_fig, "05_keymetrics_by_segment")
    plt.show()

    print(f"Complete-history comparison: {scan_data['PersonId'].nunique():,} people")
    print(f"Metrics included ({len(scan_metrics)}): {', '.join(scan_metrics)}")
    if missing_scan_metrics:
        print(
            f"Metrics not present in this export ({len(missing_scan_metrics)}): "
            f"{', '.join(missing_scan_metrics)}"
        )
    skipped_segments = [s for s in segment_order if s not in reported_segments]
    if skipped_segments:
        print(
            f"Segments below the {MIN_DISPLAY_N}-person floor and therefore not "
            f"reported: {', '.join(skipped_segments)}"
        )
    print()
    print(segment_scan_table.to_string(index=False))

# Indexed lookup used by the narrative sections below. Reading through lookup() means a
# segment that fell below the reporting floor produces "not available" rather than an error.
scan_by_segment = (
    segment_scan_table.set_index("UsageSegment_12w")
    if segment_scan_table is not None else None
)


## 6. Adjusted and process-level diagnostics

The native key-metrics scan is the primary segment comparison. This section adds two
diagnostics that the package scan does not provide:

1. **Adjusted differences.** How Power and Habitual Users compare with everyone else on
   each collaboration metric, controlling for function and manager status. Results are in
   standard deviations, with 95% confidence intervals, so that metrics on different
   scales can be read on one axis.
2. **Process indicators.** Median meeting length, after-hours share, meeting multitasking,
   and focus realisation, each shown in its own units (minutes, or a percentage) for each
   journey stage. These are deliberately not expressed as a percentage difference from the
   Non-user median: three of the four indicators are already percentages, and a percentage
   change in a percentage is easy to misread. Comparing the actual medians side by side
   needs no such translation.

The two views answer different questions. The adjusted chart asks whether a gap survives
once role is accounted for; the process panels show the plain, unadjusted levels a reader
can sanity-check against their own experience. Read them together.

A note on size. Significance and importance are not the same thing, and a large person
count makes small differences statistically detectable. The printed output flags which
adjusted differences also reach the `MATERIAL_EFFECT_SD` floor, which Section 7 explains
in full.

These remain observational associations rather than Copilot effects.

In [ ]:
recent_start = latest_week - pd.Timedelta(weeks=RECENT_WEEKS - 1)
recent = observed[observed["MetricDate"] >= recent_start].copy()
recent = recent.merge(latest_segments, on="PersonId", how="inner")

process_source_metrics = [
    "Meetings", "Meeting_hours", "Collaboration_hours",
    "After_hours_collaboration_hours", "Multitasking_hours",
    "Available_to_focus_hours", "Uninterrupted_hours",
]
for metric in process_source_metrics:
    if metric not in recent.columns:
        recent[metric] = np.nan


def safe_rate(numerator, denominator, scale):
    return np.where(denominator > 0, numerator / denominator * scale, np.nan)


recent["meeting_length_min"] = safe_rate(
    recent["Meeting_hours"], recent["Meetings"], 60
)
recent["after_hours_share_pct"] = safe_rate(
    recent["After_hours_collaboration_hours"], recent["Collaboration_hours"], 100
)
recent["meeting_multitask_share_pct"] = safe_rate(
    recent["Multitasking_hours"], recent["Meeting_hours"], 100
)
recent["focus_realisation_pct"] = safe_rate(
    recent["Uninterrupted_hours"], recent["Available_to_focus_hours"], 100
)

ratio_labels = {
    "meeting_length_min": "Meeting length (minutes)",
    "after_hours_share_pct": "After-hours share of collaboration",
    "meeting_multitask_share_pct": "Meeting time spent multitasking",
    "focus_realisation_pct": "Available focus time uninterrupted",
}
# Keep only the ratios this export can actually support.
ratio_labels = {
    key: label for key, label in ratio_labels.items()
    if recent[key].notna().any()
}
ratio_cols = list(ratio_labels)

process_summary = None
if ratio_cols:
    person_ratios = (
        recent.groupby(["PersonId", "JourneyStage"])[ratio_cols].mean().reset_index()
    )
    stage_counts = person_ratios["JourneyStage"].value_counts()
    reportable_stages = [
        stage for stage in journey_stage_order
        if stage_counts.get(stage, 0) >= MIN_PRIVACY_N
    ]
    process_summary = (
        person_ratios[person_ratios["JourneyStage"].isin(reportable_stages)]
        .groupby("JourneyStage")[ratio_cols].median()
        .reindex(journey_stage_order)
        .reset_index()
    )
    save_table(process_summary, "06_process_ratio_medians")
else:
    print(
        "None of the process ratios can be computed from this export, because the "
        "underlying meeting, focus, or after-hours metrics are absent."
    )

level_metrics = {
    "Collaboration_hours": "Collaboration hours",
    "Meetings": "Meetings",
    "Meeting_hours": "Meeting hours",
    "Chats_sent": "Chats sent",
    "Emails_sent": "Emails sent",
    "Active_connected_hours": "Active connected hours",
    "Multitasking_hours": "Multitasking hours",
    "Available_to_focus_hours": "Available-to-focus hours",
    "Uninterrupted_hours": "Uninterrupted hours",
    "After_hours_collaboration_hours": "After-hours collaboration",
}
level_metrics = {
    metric: label for metric, label in level_metrics.items()
    if metric in recent.columns and recent[metric].notna().any()
}

adjusted = None
if not level_metrics:
    print("No collaboration level metrics are available, so the adjusted model is skipped.")
else:
    person_levels = (
        recent.groupby(["PersonId", "JourneyStage"])[list(level_metrics)].mean().reset_index()
        .merge(latest_attributes[["PersonId", "FunctionType", "ManagerStatus"]],
               on="PersonId", how="left")
    )
    person_levels["power_habitual"] = (
        person_levels["JourneyStage"] == "Power + Habitual"
    ).astype(int)
    person_levels["FunctionType"] = person_levels["FunctionType"].fillna("Missing")
    person_levels["ManagerStatus"] = person_levels["ManagerStatus"].fillna("Unknown")

    # Only control for attributes that actually vary, otherwise the dummy block is
    # perfectly collinear with the intercept.
    control_columns = [
        column for column in ["FunctionType", "ManagerStatus"]
        if person_levels[column].nunique() > 1
    ]
    control_note = (
        f"controlling for {' and '.join(control_columns)}" if control_columns
        else "unadjusted, because no organizational attribute varies in this export"
    )
    parts = [person_levels[["power_habitual"]]]
    if control_columns:
        parts.append(pd.get_dummies(person_levels[control_columns],
                                    drop_first=True, dtype=float))
    controls = sm.add_constant(pd.concat(parts, axis=1).astype(float))

    adjusted_rows = []
    for metric, label in level_metrics.items():
        ok = person_levels[metric].notna()
        y = person_levels.loc[ok, metric].astype(float)
        if ok.sum() < 2 * MIN_DISPLAY_N or y.std() == 0:
            continue
        if person_levels.loc[ok, "power_habitual"].nunique() < 2:
            continue
        y_z = (y - y.mean()) / y.std()
        model = sm.OLS(y_z, controls.loc[ok]).fit(cov_type="HC3")
        beta = float(model.params["power_habitual"])
        se = float(model.bse["power_habitual"])
        adjusted_rows.append({
            "metric": metric,
            "label": label,
            "n": int(ok.sum()),
            "adjusted_difference_sd": beta,
            "ci_low": beta - 1.96 * se,
            "ci_high": beta + 1.96 * se,
            "p_value": float(model.pvalues["power_habitual"]),
        })

    if not adjusted_rows:
        print(
            "No metric had enough variation and sample size to fit the adjusted model, "
            "so this comparison is skipped."
        )
    else:
        adjusted = pd.DataFrame(adjusted_rows).sort_values("adjusted_difference_sd")
        save_table(adjusted, "06_adjusted_working_pattern_associations")

ratio_units = {
    "meeting_length_min": ("minutes", "{:.0f}"),
    "after_hours_share_pct": ("% of collaboration", "{:.1f}"),
    "meeting_multitask_share_pct": ("% of meeting time", "{:.1f}"),
    "focus_realisation_pct": ("% of available focus time", "{:.1f}"),
}

if adjusted is not None:
    fig, ax = plt.subplots(figsize=(9.5, max(3.4, 0.45 * len(adjusted) + 1.8)))
    ypos = np.arange(len(adjusted))
    ax.errorbar(
        adjusted["adjusted_difference_sd"], ypos,
        xerr=[
            adjusted["adjusted_difference_sd"] - adjusted["ci_low"],
            adjusted["ci_high"] - adjusted["adjusted_difference_sd"],
        ],
        fmt="o", color=journey_stage_color_map["Power + Habitual"], ecolor=C_GREY,
        capsize=3,
    )
    ax.axvline(0, color=C_TEXT, lw=1)
    ax.set_yticks(ypos)
    ax.set_yticklabels(adjusted["label"])
    ax.set_xlabel("Adjusted difference (standard deviations)")
    ax.set_title("Power and Habitual Users versus others, adjusted",
                 loc="left", fontweight="bold")
    add_subtitle(ax, f"OLS on person-level weekly means, {control_note}; bars are 95% CI")
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    save_figure(fig, "06_adjusted_working_patterns")
    plt.show()

# Process indicators are shown in their own units, one panel per indicator, rather than
# as a percentage difference from the Non-user median. A relative view of an indicator
# that is itself a percentage produces a percentage of a percentage, which is easy to
# misread; the actual medians are directly comparable and need no explanation.
if process_summary is not None and ratio_cols:
    plot_ratios = [
        column for column in ratio_cols
        if process_summary[column].notna().sum() >= 2
    ]
    if plot_ratios:
        n_panels = len(plot_ratios)
        fig, axes = plt.subplots(1, n_panels, figsize=(3.6 * n_panels, 4.3),
                                 squeeze=False)
        stage_values = process_summary.set_index("JourneyStage")
        stages = [s for s in journey_stage_order if s in stage_values.index]
        for ax, column in zip(axes[0], plot_ratios):
            unit, number_format = ratio_units.get(column, ("", "{:.1f}"))
            values = [stage_values.loc[stage, column] for stage in stages]
            bars = ax.bar(
                range(len(stages)), values, width=0.62,
                color=[journey_stage_color_map[stage] for stage in stages],
            )
            ax.set_xticks(range(len(stages)))
            ax.set_xticklabels(
                [stage.replace(" + ", " +\n") for stage in stages], fontsize=8.5
            )
            ax.set_title(ratio_labels[column], loc="left", fontweight="bold", fontsize=10)
            ax.set_ylabel(unit, fontsize=8.5)
            top = np.nanmax(values) if np.isfinite(np.nanmax(values)) else 1
            ax.set_ylim(0, top * 1.22)
            ax.grid(axis="x", visible=False)
            for bar, value in zip(bars, values):
                if pd.notna(value):
                    ax.annotate(
                        number_format.format(value),
                        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=9,
                    )
        fig.suptitle(
            f"Median process indicators by journey stage, last {RECENT_WEEKS} weeks",
            x=0.01, ha="left", fontsize=13, fontweight="bold",
        )
        fig.text(
            0.01, -0.04,
            "Each panel is a median across people, shown in its own units. These are "
            "unadjusted group medians, so read them alongside the adjusted chart above.",
            fontsize=8.5, color=C_GREY,
        )
        fig.tight_layout()
        save_figure(fig, "06_process_indicators")
        plt.show()

if adjusted is not None:
    print(f"Adjusted Power + Habitual associations ({control_note}):")
    print(adjusted[["label", "n", "adjusted_difference_sd", "ci_low", "ci_high",
                    "p_value"]].to_string(index=False, formatters={
        "adjusted_difference_sd": "{:+.3f}".format,
        "ci_low": "{:+.3f}".format,
        "ci_high": "{:+.3f}".format,
        "p_value": "{:.3g}".format,
    }))
    significant = adjusted[adjusted["p_value"] < 0.05]
    material = significant[
        significant["adjusted_difference_sd"].abs() >= MATERIAL_EFFECT_SD
    ]
    print(
        f"\n{len(significant)} of {len(adjusted)} metrics differ significantly at the "
        f"5% level after adjustment, and {len(material)} of those reach the "
        f"{MATERIAL_EFFECT_SD} SD materiality floor. Metrics that do neither are "
        "reported above rather than dropped, so that null results stay visible."
    )

if process_summary is not None:
    print(f"\nMedian process indicators over the latest {RECENT_WEEKS} weeks:")
    print(process_summary.to_string(index=False))

# Indexed lookups used by the narrative sections below.
process_by_stage = (
    process_summary.set_index("JourneyStage") if process_summary is not None else None
)
adjusted_by_metric = adjusted.set_index("metric") if adjusted is not None else None


## 7. Same-person check: what changes in heavier Copilot-use weeks?

This view compares each person with themselves and removes common week effects. It controls
for stable individual differences such as role propensity, but it still cannot distinguish
Copilot effects from unusually demanding weeks.

The coefficient is the standard-deviation change in the process indicator associated with
a one-standard-deviation increase in `log(1 + actions)`.

### How to read this, and when to conclude anything

Two tests have to be passed before a result here is worth acting on, and they are
different questions.

**1. Is it distinguishable from zero?** The horizontal bar is the 95% confidence interval.
If any part of it touches the zero line, the data cannot tell the difference between the
observed association and no association at all. It does not matter how far left or right
the dot sits: an interval that crosses zero is a null result, and should be reported as
one.

**2. Is it big enough to matter?** This is the test that is easy to skip. These models run
on tens of thousands of person-weeks, and with a sample that large almost anything becomes
statistically significant eventually. A coefficient of `-0.01` SD can have a confidence
interval comfortably clear of zero while describing a change far too small to notice, let
alone manage. The shaded band on the chart marks the region below `MATERIAL_EFFECT_SD`
(0.1 standard deviations by default), which is the conventional floor for a small effect.
A result inside that band is real but negligible.

So, to answer the question directly: conclude that heavier Copilot use is associated with
lower after-hours work only when the **entire** interval sits left of zero **and** the dot
sits outside the shaded band. If the interval crosses zero, the honest description is "no
detectable association", however suggestive the direction looks.

The cell below labels each result against both tests, so the verdict does not depend on
eyeballing the chart.

**And even then it is not causal.** A material, statistically clear coefficient here still
only says that heavier-use weeks look different from lighter-use weeks for the same person.
Busy weeks plausibly drive both the Copilot usage and the process change. Establishing
direction needs a design built for it, such as the
[event-study and difference-in-differences examples](https://microsoft.github.io/viva-insights-sample-code/copilot/#event-study--difference-in-differences)
or the [Copilot Causal Toolkit](https://microsoft.github.io/viva-insights-sample-code/copilot-causal-toolkit/).


In [ ]:
within_data = observed.copy()
within_data["Total_Copilot_actions_taken"] = (
    within_data["Total_Copilot_actions_taken"].fillna(0)
)
for metric in process_source_metrics:
    if metric not in within_data.columns:
        within_data[metric] = np.nan
within_data["meeting_length_min"] = safe_rate(
    within_data["Meeting_hours"], within_data["Meetings"], 60
)
within_data["after_hours_share_pct"] = safe_rate(
    within_data["After_hours_collaboration_hours"],
    within_data["Collaboration_hours"], 100
)
within_data["meeting_multitask_share_pct"] = safe_rate(
    within_data["Multitasking_hours"], within_data["Meeting_hours"], 100
)
within_data["focus_realisation_pct"] = safe_rate(
    within_data["Uninterrupted_hours"], within_data["Available_to_focus_hours"], 100
)


def two_way_demean(values, person, week, iterations=10):
    result = values.astype(float).copy()
    for _ in range(iterations):
        result = result - result.groupby(person).transform("mean")
        result = result - result.groupby(week).transform("mean")
    return result


within_rows = []
for metric, label in ratio_labels.items():
    work = within_data[[
        "PersonId", "MetricDate", "Total_Copilot_actions_taken", metric
    ]].dropna().copy()
    if work.empty:
        continue
    work["log_actions"] = np.log1p(work["Total_Copilot_actions_taken"])
    work["x_within"] = two_way_demean(
        work["log_actions"], work["PersonId"], work["MetricDate"]
    )
    work["y_within"] = two_way_demean(
        work[metric], work["PersonId"], work["MetricDate"]
    )
    x_sd, y_sd = work["x_within"].std(), work["y_within"].std()
    if len(work) < 2 * MIN_DISPLAY_N or pd.isna(x_sd) or pd.isna(y_sd) or x_sd == 0 or y_sd == 0:
        continue
    model = sm.OLS(
        work["y_within"] / y_sd,
        sm.add_constant(work["x_within"] / x_sd),
    ).fit(cov_type="cluster", cov_kwds={"groups": work["PersonId"]})
    beta = float(model.params["x_within"])
    se = float(model.bse["x_within"])
    within_rows.append({
        "metric": metric,
        "label": label,
        "n_person_weeks": len(work),
        "beta_sd": beta,
        "ci_low": beta - 1.96 * se,
        "ci_high": beta + 1.96 * se,
        "p_value": float(model.pvalues["x_within"]),
    })

within_results = None
if not within_rows:
    print(
        "No process ratio has enough person-week observations for the same-person "
        "check, so this section is skipped."
    )
else:
    within_results = pd.DataFrame(within_rows).sort_values("beta_sd")
    # Classify each result against both tests: is it distinguishable from zero, and is
    # it large enough to matter. The verdict is stored so that the chart, the printed
    # narrative, and the executive summary cannot disagree with one another.
    crosses_zero = (within_results["ci_low"] <= 0) & (within_results["ci_high"] >= 0)
    is_material = within_results["beta_sd"].abs() >= MATERIAL_EFFECT_SD
    within_results["verdict"] = np.select(
        [crosses_zero, ~crosses_zero & ~is_material],
        ["No detectable association", "Detectable but negligible"],
        default="Detectable and material",
    )
    within_results["direction"] = np.where(
        within_results["beta_sd"] > 0, "higher", "lower"
    )
    save_table(within_results, "07_within_person_process_associations")

    verdict_style = {
        "Detectable and material": dict(color=C_TEAL, mfc=C_TEAL),
        "Detectable but negligible": dict(color=C_TEAL, mfc="white"),
        "No detectable association": dict(color=C_GREY, mfc="white"),
    }

    fig, ax = plt.subplots(figsize=(10.5, max(3.0, 0.62 * len(within_results) + 2.4)))
    ypos = np.arange(len(within_results))

    # Region of practical negligibility: anything landing inside it is too small to act
    # on, whatever its p-value.
    ax.axvspan(-MATERIAL_EFFECT_SD, MATERIAL_EFFECT_SD, color=C_LIGHT, zorder=0)
    ax.axvline(0, color=C_TEXT, lw=1.2, zorder=1)

    for y, row in zip(ypos, within_results.itertuples()):
        style = verdict_style[row.verdict]
        ax.errorbar(
            row.beta_sd, y,
            xerr=[[row.beta_sd - row.ci_low], [row.ci_high - row.beta_sd]],
            fmt="o", capsize=4, lw=1.8, markersize=7,
            color=style["color"], ecolor=style["color"], markerfacecolor=style["mfc"],
            zorder=3,
        )

    ax.set_yticks(ypos)
    ax.set_yticklabels(within_results["label"])
    ax.set_ylim(-0.7, len(within_results) - 0.3)
    ax.set_xlabel("Same-person, week-adjusted association (standard deviations)")
    ax.set_title("Process indicators in heavier Copilot-use weeks", loc="left",
                 fontweight="bold")
    add_subtitle(ax, "Change per 1 SD increase in log(1 + weekly Copilot actions), person and week effects removed")
    ax.grid(axis="y", visible=False)

    # Keep the negligibility band visible even when every interval is tiny.
    span = max(
        float(within_results["ci_high"].abs().max()),
        float(within_results["ci_low"].abs().max()),
        MATERIAL_EFFECT_SD * 1.6,
    )
    ax.set_xlim(-span * 1.35, span * 1.35)

    for y, row in zip(ypos, within_results.itertuples()):
        ax.annotate(
            f"{row.beta_sd:+.3f} SD  [{row.ci_low:+.3f}, {row.ci_high:+.3f}]",
            (ax.get_xlim()[1], y), xytext=(-6, 0), textcoords="offset points",
            ha="right", va="center", fontsize=7.5, color=C_GREY,
        )

    legend_handles = [
        plt.Line2D([], [], marker="o", linestyle="none", markersize=7,
                   color=style["color"], markerfacecolor=style["mfc"], label=verdict)
        for verdict, style in verdict_style.items()
        if verdict in set(within_results["verdict"])
    ]
    legend_handles.append(
        plt.Rectangle((0, 0), 1, 1, color=C_LIGHT,
                      label=f"Negligible zone (< {MATERIAL_EFFECT_SD} SD)")
    )
    ax.legend(handles=legend_handles, frameon=False, fontsize=8,
              loc="upper left", bbox_to_anchor=(0, -0.16), ncol=4,
              handletextpad=0.5, columnspacing=1.4)
    fig.tight_layout()
    save_figure(fig, "07_within_person_process")
    plt.show()

    print(within_results[[
        "label", "n_person_weeks", "beta_sd", "ci_low", "ci_high", "p_value", "verdict",
    ]].to_string(index=False, formatters={
        "beta_sd": "{:+.3f}".format,
        "ci_low": "{:+.3f}".format,
        "ci_high": "{:+.3f}".format,
        "p_value": "{:.3g}".format,
    }))

    print("\nWhat each result supports:")
    for row in within_results.itertuples():
        if row.verdict == "No detectable association":
            print(
                f"  {row.label}: no detectable association. The 95% interval "
                f"[{row.ci_low:+.3f}, {row.ci_high:+.3f}] includes zero, so the "
                f"{row.direction} direction of the point estimate is not supported."
            )
        elif row.verdict == "Detectable but negligible":
            print(
                f"  {row.label}: statistically clear but negligible. The interval "
                f"excludes zero, yet {row.beta_sd:+.3f} SD sits inside the "
                f"{MATERIAL_EFFECT_SD} SD negligible zone, which is a difference too "
                "small to manage against."
            )
        else:
            print(
                f"  {row.label}: {row.direction} in heavier-use weeks, by "
                f"{abs(row.beta_sd):.3f} SD [{row.ci_low:+.3f}, {row.ci_high:+.3f}]. "
                "This clears both the significance and the materiality bar, though it "
                "remains an association rather than an effect."
            )

    material_count = int((within_results["verdict"] == "Detectable and material").sum())
    if material_count == 0:
        print(
            "\nNo indicator clears both bars. On this data, the honest headline is that "
            "heavier Copilot-use weeks do not look meaningfully different from lighter "
            "weeks for the same person, and none of these directions should be reported "
            "as a finding."
        )
    else:
        print(
            f"\n{material_count} of {len(within_results)} indicators clear both the "
            "significance and materiality bars. Treat the rest as null results."
        )

within_by_metric = (
    within_results.set_index("metric") if within_results is not None else None
)


## 8. Evidence-led story synthesis

This section translates the analytical outputs into a reusable evidence matrix for
executive reporting. Each insight records the supporting numbers, the interpretation,
and the principal caveat.

The synthesis deliberately distinguishes **information-exchange intensity** from
information-flow speed or network breadth. A Person Query contains no direct measure of
information velocity, and collaboration-network metrics are only available if your query
included them, so the wording below stays within what the data can support.

Each insight is assembled from the values computed above rather than written in advance,
so the direction of every statement follows your data. Where an input is missing, the
insight is omitted rather than asserted.

In [ ]:
segment_counts = segment_summary.set_index("segment")


def describe(value, rising, falling, flat, tolerance=1.0):
    """Pick wording from the computed value, so the narrative follows the data."""
    if pd.isna(value):
        return flat
    if value > tolerance:
        return rising
    if value < -tolerance:
        return falling
    return flat


def relative_gap(metric, group="Power User", reference="Non-user"):
    """Percentage difference between two segments, or NaN when either is unreported."""
    group_value = lookup(scan_by_segment, group, metric)
    reference_value = lookup(scan_by_segment, reference, metric)
    if pd.isna(group_value) or pd.isna(reference_value) or not reference_value:
        return np.nan
    return 100 * (group_value / reference_value - 1)


gap_metrics = {
    "Collaboration_hours": "Collaboration hours",
    "Work_session_span_hours": "Work-session span hours",
    "Active_connected_hours": "Active connected hours",
    "Meetings": "Meetings",
    "Meeting_hours": "Meeting hours",
    "Calls": "Calls",
    "Chats_sent": "Chats sent",
    "Emails_sent": "Emails sent",
    "Multitasking_hours": "Multitasking hours",
    "Available_to_focus_hours": "Available-to-focus hours",
    "Uninterrupted_hours": "Uninterrupted hours",
    "After_hours_collaboration_hours": "After-hours collaboration hours",
    "Time_with_leadership": "Time with leadership",
}
if scan_by_segment is not None:
    gap_metrics = {
        metric: label for metric, label in gap_metrics.items()
        if metric in scan_by_segment.columns
    }
    gap_rows = []
    for metric, label in gap_metrics.items():
        for group in ["Power User", "Habitual User"]:
            gap_rows.append({
                "metric": metric,
                "label": label,
                "segment": group,
                "segment_value": lookup(scan_by_segment, group, metric),
                "non_user_value": lookup(scan_by_segment, "Non-user", metric),
                "difference_pct": relative_gap(metric, group=group),
            })
    segment_gap_table = pd.DataFrame(gap_rows)
    save_table(segment_gap_table, "08_segment_gaps_vs_non_users")
else:
    segment_gap_table = None
    gap_metrics = {}

sustained_n = int(
    segment_counts.loc["Power User", "people"]
    + segment_counts.loc["Habitual User", "people"]
)
sustained_pct = pct(sustained_n, int(segment_summary["people"].sum()))
novice_pct = float(segment_counts.loc["Novice User", "pct"])

manager_lookup = (
    manager_summary.set_index("ManagerStatus") if manager_summary is not None else None
)
manager_sustained = lookup(manager_lookup, "Manager", "power_habitual_pct")
ic_sustained = lookup(manager_lookup, "IC", "power_habitual_pct")

insights = []

# 1. Scale: coverage growth against depth growth.
coverage_delta = last_row["coverage_pct"] - first_row["coverage_pct"]
depth_delta_pct = (
    100 * (last_row["actions_per_active"] / first_row["actions_per_active"] - 1)
    if first_row["actions_per_active"] else np.nan
)
insights.append({
    "theme": "Scale",
    "insight": (
        "Copilot metric coverage "
        + describe(coverage_delta, "expanded", "contracted", "held steady")
        + ", while depth among active users "
        + describe(depth_delta_pct, "rose", "fell", "stayed broadly stable", tolerance=5)
        + "."
    ),
    "evidence": (
        f"Coverage moved from {fmt(first_row['coverage_pct'])}% to "
        f"{fmt(last_row['coverage_pct'])}%; actions per active user moved from "
        f"{fmt(first_row['actions_per_active'])} to "
        f"{fmt(last_row['actions_per_active'])} ({fmt(depth_delta_pct, '{:+.0f}')}%)."
    ),
    "implication": describe(
        coverage_delta - (depth_delta_pct if pd.notna(depth_delta_pct) else 0),
        "The near-term opportunity is activation and habit formation rather than adding more covered users.",
        "Depth is growing faster than reach, so widening coverage is the larger remaining opportunity.",
        "Reach and depth are moving together, so no single lever stands out.",
        tolerance=5,
    ),
    "caveat": (
        f"Licensing is measured directly from '{LICENSE_COL}'." if has_license_col
        else "Metric coverage is a proxy, because no enabled-days column is available."
    ),
})

# 2. Habit: the size of the conversion pool.
insights.append({
    "theme": "Habit",
    "insight": (
        f"The Novice population is "
        + describe(novice_pct - sustained_pct,
                   "larger than the sustained-user base, so there is substantial headroom",
                   "smaller than the sustained-user base, so adoption is already consolidating",
                   "comparable to the sustained-user base")
        + "."
    ),
    "evidence": (
        f"{fmt(sustained_pct)}% are Power or Habitual Users, while {fmt(novice_pct)}% "
        "are Novice Users."
    ),
    "implication": "Targeted use-case reinforcement is the lever that moves Novice Users into repeat use.",
    "caveat": "Recent entrants have incomplete rolling histories and may be understated.",
})

# 3. Onboarding: only when at least two cohorts clear the display floor.
if len(plot_cohorts) >= 2:
    oldest_cohort, newest_cohort = plot_cohorts.iloc[0], plot_cohorts.iloc[-1]
    cohort_gap = oldest_cohort["active_latest_pct"] - newest_cohort["active_latest_pct"]
    insights.append({
        "theme": "Onboarding",
        "insight": (
            "Newly covered cohorts activate "
            + describe(cohort_gap, "more slowly than established cohorts",
                       "faster than established cohorts",
                       "at a similar rate to established cohorts", tolerance=5)
            + "."
        ),
        "evidence": (
            f"The earliest qualifying cohort is {fmt(oldest_cohort['active_latest_pct'])}% "
            f"active in the latest week versus "
            f"{fmt(newest_cohort['active_latest_pct'])}% for the newest."
        ),
        "implication": "Onboarding is best measured as a cohort conversion journey rather than a one-time launch.",
        "caveat": "The coverage date may reflect licensing, query scope, or telemetry availability rather than a rollout.",
    })

# 4. Leadership: only when the export distinguishes managers from ICs.
if pd.notna(manager_sustained) and pd.notna(ic_sustained):
    manager_gap = manager_sustained - ic_sustained
    insights.append({
        "theme": "Leadership",
        "insight": (
            "Sustained adoption is "
            + describe(manager_gap, "higher among managers", "higher among individual contributors",
                       "similar across managers and individual contributors", tolerance=2)
            + "."
        ),
        "evidence": (
            f"{fmt(manager_sustained)}% of managers are Power or Habitual Users versus "
            f"{fmt(ic_sustained)}% of individual contributors."
        ),
        "implication": describe(
            manager_gap,
            "Managers can sponsor adoption, and individual-contributor use cases need deliberate reinforcement.",
            "Adoption is bottom-up, so manager enablement is the gap to close.",
            "Enablement can be designed for one audience rather than split by seniority.",
            tolerance=2,
        ),
        "caveat": "Manager roles are structurally more collaboration intensive, which confounds the comparison.",
    })

# 5. Functional variation: only when create_rank produced a function ranking.
if rank_sustained is not None:
    function_rank = rank_sustained[rank_sustained["hrvar"] == "FunctionType"]
    if len(function_rank) >= 2:
        top_function, bottom_function = function_rank.iloc[0], function_rank.iloc[-1]
        spread = top_function["metric"] - bottom_function["metric"]
        insights.append({
            "theme": "Functional variation",
            "insight": (
                "Power + Habitual adoption "
                + describe(spread, "varies widely by function", "varies widely by function",
                           "is fairly even across functions", tolerance=10)
                + "."
            ),
            "evidence": (
                f"{top_function['attributes']} leads qualifying functions at "
                f"{fmt(top_function['metric'])}% versus {bottom_function['attributes']} "
                f"at {fmt(bottom_function['metric'])}%, a spread of "
                f"{fmt(spread, '{:.0f}')} points."
            ),
            "implication": describe(
                spread,
                "The next enablement wave should be role-specific rather than enterprise-generic.",
                "The next enablement wave should be role-specific rather than enterprise-generic.",
                "A single enterprise-wide enablement approach is defensible here.",
                tolerance=10,
            ),
            "caveat": "These are descriptive group differences, not performance rankings.",
        })

# 6 to 8. Segment gaps, only for metrics the scan actually reported.
if scan_by_segment is not None:
    collab_gap = relative_gap("Collaboration_hours")
    if pd.notna(collab_gap):
        insights.append({
            "theme": "Information exchange",
            "insight": (
                "Power Users operate in a "
                + describe(collab_gap, "more", "less", "similarly", tolerance=5)
                + " collaboration-intensive environment than Non-users."
            ),
            "evidence": (
                f"Power Users average "
                f"{fmt(lookup(scan_by_segment, 'Power User', 'Collaboration_hours'))} "
                f"collaboration hours versus "
                f"{fmt(lookup(scan_by_segment, 'Non-user', 'Collaboration_hours'))} for "
                f"Non-users ({fmt(collab_gap, '{:+.0f}')}%)."
            ),
            "implication": "Copilot is most embedded where the volume of information exchange is highest.",
            "caveat": "The data measures activity volume, not information-flow speed or quality.",
        })

    span_gap = relative_gap("Work_session_span_hours")
    if pd.notna(span_gap):
        insights.append({
            "theme": "Workday intensity",
            "insight": (
                "Higher usage coincides with "
                + describe(span_gap, "longer", "shorter", "similar", tolerance=3)
                + " work-session spans."
            ),
            "evidence": (
                f"Power Users average "
                f"{fmt(lookup(scan_by_segment, 'Power User', 'Work_session_span_hours'))} "
                f"work-session span hours, {fmt(span_gap, '{:+.0f}')}% versus Non-users."
            ),
            "implication": "Copilot adoption is concentrated in demanding roles and work patterns.",
            "caveat": "Work-session span is an hours metric, not network breadth.",
        })

    multitask_gap = relative_gap("Multitasking_hours")
    if pd.notna(multitask_gap):
        insights.append({
            "theme": "Focus",
            "insight": (
                "Multitasking is "
                + describe(multitask_gap, "higher", "lower", "comparable", tolerance=5)
                + " among Power Users than Non-users."
            ),
            "evidence": (
                f"Power Users average "
                f"{fmt(lookup(scan_by_segment, 'Power User', 'Multitasking_hours'))} "
                f"multitasking hours versus "
                f"{fmt(lookup(scan_by_segment, 'Non-user', 'Multitasking_hours'))} "
                f"({fmt(multitask_gap, '{:+.0f}')}%)."
            ),
            "implication": "Copilot enablement is best paired with meeting, asynchronous-work, and focus-time practices.",
            "caveat": "High-demand weeks may drive both Copilot use and fragmentation.",
        })

# 9. After-hours: absolute hours against share of collaboration.
after_hours_gap = relative_gap("After_hours_collaboration_hours")
power_share = lookup(process_by_stage, "Power + Habitual", "after_hours_share_pct")
non_share = lookup(process_by_stage, "Non-user", "after_hours_share_pct")
if pd.notna(after_hours_gap) and pd.notna(power_share) and pd.notna(non_share):
    share_gap = power_share - non_share
    adjusted_after_hours = lookup(
        adjusted_by_metric, "After_hours_collaboration_hours", "adjusted_difference_sd"
    )
    adjusted_p = lookup(adjusted_by_metric, "After_hours_collaboration_hours", "p_value")
    insights.append({
        "theme": "After-hours",
        "insight": (
            "After-hours collaboration as a share of total collaboration is "
            + describe(share_gap, "higher", "lower", "similar", tolerance=2)
            + " for sustained users, even though absolute after-hours hours are "
            + describe(after_hours_gap, "higher", "lower", "comparable", tolerance=5)
            + "."
        ),
        "evidence": (
            f"Absolute after-hours collaboration differs by "
            f"{fmt(after_hours_gap, '{:+.0f}')}%, while the median after-hours share is "
            f"{fmt(power_share)}% for Power and Habitual Users versus {fmt(non_share)}% "
            "for Non-users."
        ),
        "implication": describe(
            share_gap,
            "The additional load is spilling beyond the normal working pattern and is worth monitoring.",
            "The additional collaboration load sits inside the normal working pattern.",
            "The additional collaboration load sits inside the normal working pattern.",
            tolerance=2,
        ),
        "caveat": (
            f"After adjusting for role attributes the difference is "
            f"{fmt(adjusted_after_hours, '{:+.3f}')} SD (p={fmt(adjusted_p, '{:.2f}')})."
            if pd.notna(adjusted_after_hours)
            else "The adjusted model did not cover this metric."
        ),
    })

# 10. Meetings: level difference against the same-person coefficient.
power_meeting_len = lookup(process_by_stage, "Power + Habitual", "meeting_length_min")
non_meeting_len = lookup(process_by_stage, "Non-user", "meeting_length_min")
within_meeting = lookup(within_by_metric, "meeting_length_min", "beta_sd")
within_meeting_verdict = lookup(
    within_by_metric, "meeting_length_min", "verdict", default=None
)
if pd.notna(power_meeting_len) and pd.notna(non_meeting_len):
    meeting_gap = power_meeting_len - non_meeting_len
    same_person_note = ""
    if pd.notna(within_meeting) and within_meeting_verdict:
        same_person_note = (
            f"; the same-person coefficient is {fmt(within_meeting, '{:+.3f}')} SD "
            f"({within_meeting_verdict.lower()})"
        )
    insights.append({
        "theme": "Meetings",
        "insight": (
            "Copilot use is associated with "
            + describe(meeting_gap, "longer", "shorter", "similar", tolerance=2)
            + " meetings."
        ),
        "evidence": (
            f"Median meeting length is {fmt(power_meeting_len)} minutes for Power and "
            f"Habitual Users and {fmt(non_meeting_len)} minutes for Non-users"
            f"{same_person_note}."
        ),
        "implication": describe(
            -meeting_gap,
            "Meeting recap and asynchronous follow-through may already be compressing meetings.",
            "Meeting recap and asynchronous follow-through have not yet translated into shorter meetings.",
            "Meeting recap and asynchronous follow-through have not yet translated into measurable meeting compression.",
            tolerance=2,
        ),
        "caveat": "Meeting duration is inferred from weekly meeting hours divided by meeting count.",
    })

insight_evidence = pd.DataFrame(insights)
insight_evidence.insert(0, "number", range(1, len(insight_evidence) + 1))
save_table(insight_evidence, "08_insight_evidence")

print(f"Evidence-led insights generated from this dataset: {len(insight_evidence)}")
print(insight_evidence[["number", "theme", "insight", "evidence"]].to_string(index=False))


## 9. Executive interpretation

The final cell writes an executive summary grounded only in results calculated above.
It leads with the adoption journey, then separates reassuring signals from risks and
recommended next analyses.

In [ ]:
def section(title, lines):
    """Append a titled block, skipping it entirely when it has no content."""
    body = [line for line in lines if line]
    return ([title] + body + [""]) if body else []


summary_lines = ["COPILOT ADOPTION AND WAYS-OF-WORKING SUMMARY", "=" * 76, ""]

summary_lines += section("WHAT THE DATA COVERS", [
    f"  {df['PersonId'].nunique():,} people across {len(weeks)} weeks "
    f"({first_week.date()} to {latest_week.date()}).",
    f"  Removed {blank_rows_removed:,} blank rows from the source file."
    if blank_rows_removed else "",
    f"  Licensing is measured from '{LICENSE_COL}'." if has_license_col else
    "  No Copilot enabled-days column is present, so non-null Copilot telemetry is\n"
    "  labelled metric coverage rather than confirmed licensing.",
    f"  Usage segments use a {SEGMENT_WINDOW_WEEKS}-week window, "
    f"{SEGMENT_HABIT_WEEKS} active weeks required, Power User threshold "
    f"{POWER_THRESHOLD} weekly actions.",
    f"  Only {len(weeks)} weeks are available, fewer than the "
    f"{SEGMENT_WINDOW_WEEKS}-week window, so habit-based segments are understated."
    if len(weeks) < SEGMENT_WINDOW_WEEKS else "",
])

adoption_lines = [
    f"  Copilot metric coverage moved from {fmt(first_row['coverage_pct'])}% to "
    f"{fmt(last_row['coverage_pct'])}%.",
    f"  Weekly activation among covered people moved from "
    f"{fmt(first_row['active_pct_covered'])}% to "
    f"{fmt(last_row['active_pct_covered'])}%, while actions per active user moved from "
    f"{fmt(first_row['actions_per_active'])} to {fmt(last_row['actions_per_active'])}.",
    f"  At the latest week, {sustained_n:,} people ({fmt(sustained_pct)}%) were "
    f"Habitual or Power Users under the {SEGMENT_WINDOW_WEEKS}-week definition.",
]
if len(largest_additions):
    biggest = largest_additions.iloc[0]
    adoption_lines.append(
        f"  The largest single addition to coverage after the baseline week was "
        f"{int(biggest['newly_observed']):,} people in the week of "
        f"{pd.Timestamp(biggest['MetricDate']).date()}. Confirm what drove it before "
        "reading it as adoption."
    )
if len(plot_cohorts) >= 2:
    oldest_cohort, newest_cohort = plot_cohorts.iloc[0], plot_cohorts.iloc[-1]
    adoption_lines.append(
        f"  The earliest qualifying cohort is "
        f"{fmt(oldest_cohort['active_latest_pct'])}% active in the latest week, versus "
        f"{fmt(newest_cohort['active_latest_pct'])}% for the newest."
    )
summary_lines += section("ADOPTION JOURNEY", adoption_lines)

opportunity_lines = []
if pd.notna(manager_sustained) and pd.notna(ic_sustained):
    opportunity_lines += [
        f"  Power and Habitual User status is {fmt(manager_sustained)}% among managers "
        f"and {fmt(ic_sustained)}% among individual contributors.",
        "  Read this as both an enablement signal and a confounding warning, because",
        "  managers have heavier collaboration patterns regardless of Copilot use.",
    ]
if function_summary is not None:
    opportunity_lines.append(
        "  Function-level adoption varies. Use the function table to target onboarding "
        "and\n  role-specific scenarios, not to rank performance."
    )
if novice_by_function is not None and novice_by_function["novice_people"].max() > 0:
    top_novice = novice_by_function.iloc[0]
    opportunity_lines.append(
        f"  The largest single pool of Novice Users is {top_novice['FunctionType']} "
        f"({int(top_novice['novice_people']):,} people, "
        f"{fmt(top_novice['share_of_all_novices'], '{:.0f}')}% of all Novice Users)."
    )
summary_lines += section("WHERE THE OPPORTUNITY IS", opportunity_lines)

native_lines = []
if scan_by_segment is not None:
    native_lines.append(
        f"  The key-metrics scan compares {scan_data['PersonId'].nunique():,} people "
        f"across {len(scan_metrics)} metrics."
    )
    collab_gap = relative_gap("Collaboration_hours")
    if pd.notna(collab_gap):
        native_lines.append(
            f"  Power Users average "
            f"{fmt(lookup(scan_by_segment, 'Power User', 'Collaboration_hours'))} "
            f"collaboration hours versus "
            f"{fmt(lookup(scan_by_segment, 'Non-user', 'Collaboration_hours'))} for "
            f"Non-users ({fmt(collab_gap, '{:+.0f}')}%)."
        )
if rank_sustained is not None:
    function_rank = rank_sustained[rank_sustained["hrvar"] == "FunctionType"]
    if len(function_rank):
        leader = function_rank.iloc[0]
        native_lines.append(
            f"  {leader['attributes']} has the highest Power + Habitual adoption among "
            f"qualifying functions ({fmt(leader['metric'])}%)."
        )
summary_lines += section("NATIVE VIVA INSIGHTS VIEWS", native_lines)

ways_lines = []
if adjusted is not None:
    significant = adjusted[adjusted["p_value"] < 0.05]
    higher = significant[significant["adjusted_difference_sd"] > 0]
    lower = significant[significant["adjusted_difference_sd"] < 0]
    ways_lines.append(f"  Adjusted comparison, {control_note}.")
    if len(higher):
        ways_lines.append(
            "  Higher for Power and Habitual Users: "
            + ", ".join(higher["label"].tolist()) + "."
        )
    if len(lower):
        ways_lines.append(
            "  Lower for Power and Habitual Users: "
            + ", ".join(lower["label"].tolist()) + "."
        )
    if len(significant) == 0:
        ways_lines.append(
            "  No metric differs significantly at the 5% level once role attributes are "
            "controlled."
        )
    else:
        ways_lines.append(
            "  This shows where Copilot use is concentrated. It does not establish that"
        )
        ways_lines.append("  Copilot created those working patterns.")
if process_by_stage is not None:
    for column, label, unit in [
        ("meeting_length_min", "Median meeting length", " minutes"),
        ("after_hours_share_pct", "Median after-hours share of collaboration", "%"),
        ("meeting_multitask_share_pct", "Median meeting multitasking share", "%"),
        ("focus_realisation_pct", "Median available focus time uninterrupted", "%"),
    ]:
        power_value = lookup(process_by_stage, "Power + Habitual", column)
        non_value = lookup(process_by_stage, "Non-user", column)
        if pd.notna(power_value) and pd.notna(non_value):
            ways_lines.append(
                f"  {label}: {fmt(power_value)}{unit} for Power and Habitual Users "
                f"versus {fmt(non_value)}{unit} for Non-users."
            )
summary_lines += section("WAYS OF WORKING", ways_lines)

if within_results is not None:
    within_lines = ["  Association with a 1 SD increase in log weekly Copilot actions,",
                    "  holding the person and the week constant. A result counts only if",
                    f"  its interval clears zero and it reaches {MATERIAL_EFFECT_SD} SD:"]
    for row in within_results.itertuples():
        if row.verdict == "Detectable and material":
            note = f"{row.direction}, material"
        elif row.verdict == "Detectable but negligible":
            note = "clears zero but negligible"
        else:
            note = "no detectable association"
        within_lines.append(f"    {row.label}: {row.beta_sd:+.3f} SD ({note}).")
    material_count = int((within_results["verdict"] == "Detectable and material").sum())
    if material_count == 0:
        within_lines.append(
            "  No indicator clears both bars, so heavier-use weeks do not look"
        )
        within_lines.append(
            "  meaningfully different from lighter weeks for the same person."
        )
    within_lines.append(
        "  A same-person comparison removes stable individual differences, but it still"
    )
    within_lines.append(
        "  cannot separate a Copilot effect from an unusually demanding week."
    )
    summary_lines += section("SAME-PERSON CHECK", within_lines)

replication_lines = []
if leading_functions is not None:
    replication_lines.append(
        f"  {len(leading_functions)} function(s) already exceed "
        f"{LEADING_FUNCTION_THRESHOLD_PCT:.0f}% Power + Habitual adoption and are "
        "candidates for a replication playbook."
        if len(leading_functions) else
        f"  No function has yet crossed {LEADING_FUNCTION_THRESHOLD_PCT:.0f}% Power + "
        "Habitual adoption."
    )
summary_lines += section("WHERE TO REPLICATE", replication_lines)

action_lines = []
if len(largest_additions):
    action_lines.append(
        f"  1. Confirm whether the coverage jumps (largest in the week of "
        f"{pd.Timestamp(largest_additions.iloc[0]['MetricDate']).date()}) reflect "
        "licensing,\n     query scope, telemetry changes, or a deliberate enablement wave."
    )
action_lines += [
    f"  {len(action_lines) + 1}. Prioritise recent coverage cohorts and lower-adoption "
    "groups for role-based\n     onboarding, then track conversion to Habitual or Power "
    "status over the next window.",
]
if pd.notna(manager_sustained) and pd.notna(ic_sustained) and manager_sustained > ic_sustained:
    action_lines.append(
        f"  {len(action_lines) + 1}. Use managers as adoption sponsors, while building "
        "explicit individual-contributor\n     use cases so that adoption does not remain "
        "manager-led."
    )
if within_results is not None or process_by_stage is not None:
    action_lines.append(
        f"  {len(action_lines) + 1}. Pair Copilot enablement with meeting recap, "
        "asynchronous updates, and focus-time\n     norms, and monitor multitasking share "
        "and focus realisation as guardrail metrics."
    )
action_lines.append(
    f"  {len(action_lines) + 1}. Re-run with a longer panel before making longer-horizon "
    "retention claims. Causal\n     claims need a separate design: see "
    "https://microsoft.github.io/viva-insights-sample-code/causal-inference/"
)
summary_lines += section("RECOMMENDED ACTIONS", action_lines)

summary_lines += [
    "INTERPRETATION LIMIT",
    "  All findings are observational associations. They do not establish that Copilot",
    "  caused changes in collaboration, focus, or wellbeing.",
]

summary_text = "\n".join(summary_lines)
print(summary_text)
(OUTPUT_DIR / "executive_summary.txt").write_text(summary_text, encoding="utf-8")

print(f"\nWrote {len(TABLES)} tables, {len(FIGURES)} figures, and executive_summary.txt")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
